# 02 — Fine-tune, sinh caption và so sánh kho caption với kho lai

**Chạy sau notebook 01.** Một lần Run all cho một seed sẽ thực hiện cả hai nhánh:

| Nhánh | 5 token bộ nhớ | Khởi tạo | Fine-tune thêm |
|---|---|---|---|
| `caption_control` — đối chứng | 5 caption | L5A cùng seed | Tối đa 15 epochs, dừng sớm |
| `hybrid_3cap_2spo` — kho lai | 3 caption + 2 bộ ba | Cùng checkpoint L5A | Tối đa 15 epochs, dừng sớm |

15 epochs là trần huấn luyện, không bắt buộc chạy hết. Cả hai nhánh dùng cùng quy tắc early stopping: dừng sau 4 epoch **liên tiếp** không vượt Greedy Val CIDEr tốt nhất trước đó; nếu cải thiện, bộ đếm quay về 0. Lưu checkpoint của epoch Val tốt nhất. Vì dừng sớm, số epoch thực chạy của hai nhánh có thể khác nhau; bảng kết quả ghi rõ. Cả hai nhánh học toàn bộ captioner L5A; CLIP và RelTR đóng băng. Không thêm tham số hay tăng số token. Mỗi token tri thức là một embedding 512 chiều được chiếu qua `Linear(512,256)` hiện có. Khi không đủ bộ ba hợp lệ, lấy thêm caption còn lại để luôn đủ 5 token.

Truy hồi bộ ba: **CLIP ảnh → FAISS của các chuỗi S–P–O Train**, lấy 2 quan hệ hợp lệ; không biến quan hệ riêng của ảnh truy vấn thành mục kho. Caption vẫn dùng Semantic + MMR L5A với lambda đã chọn trên Val của seed. Bộ ba không qua MMR; phương án này kiểm tra toàn bộ cấu hình kho lai, chưa tách riêng đóng góp của từng thay đổi truy hồi.

Chọn best epoch bằng **Greedy Val CIDEr**. Sau khi cả hai nhánh train/Val xong, mặc định chấm **Karpathy Test 5.000 ảnh**, Beam=3, max_len=30, alpha=0.6, đủ 8 COCO metrics ×100. METEOR/SPICE qua Java 11. Không dùng Test để chọn kho/checkpoint, không tự xóa khóa Test.

Chạy đúng ba seed **42,44,46** khi cần bảng Mean ± Std; mặc định seed42 chỉ để tiện thao tác. Chỉ một runtime chạy một seed; khóa `/tmp` không khóa xuyên máy Colab.


## 1. Colab GPU, Google Drive và bộ chấm COCO


In [1]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
from google.colab import drive
sys.dont_write_bytecode = True
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nltk==3.9.1', 'pycocoevalcap==1.2', 'faiss-cpu'], check=True)
java_home = Path('/usr/lib/jvm/java-11-openjdk-amd64')
if not (java_home/'bin/java').is_file():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-11-jre-headless'], check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home/'bin') + ':' + os.environ.get('PATH', '')
subprocess.run(['java', '-version'], check=True)
import nltk
if nltk.__version__ != '3.9.1':
    raise RuntimeError('Restart session rồi chạy lại để nạp NLTK 3.9.1')
for corpus in ['wordnet', 'omw-1.4']:
    if not nltk.download(corpus, quiet=True, raise_on_error=True):
        raise RuntimeError('Không tải được ' + corpus)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU rồi Run all. Sinh caption 5.000 ảnh cần GPU để chạy thực tế.')

Mounted at /content/drive


## 2. Chọn seed và recipe trước khi chạy
Để chạy thăm dò trên Val, đặt `RUN_TEST=False` trước Run all. Mặc định True đáp ứng chạy đến bảng điểm Test. Hai nhánh mất thời gian train thật; đây không phải notebook ablation chỉ suy luận. Đường dẫn ghi kết quả được kiểm tra trước khi tạo thí nghiệm.


In [2]:
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
SEED = 46  # Chạy đúng 42 / 44 / 46; dùng checkpoint L5A đã train của chính seed.
EPOCHS = 15  # Số epoch tối đa, không bắt buộc chạy hết.
EARLY_STOP_PATIENCE = 4  # Dừng nếu 4 epoch liên tiếp không cải thiện Greedy Val CIDEr.
EARLY_STOP_MIN_DELTA = 0.0
BATCH_SIZE = 64
LEARNING_RATE = 5e-5
RUN_TEST = True
OUTPUT_ROOT = DATA_ROOT/'baseline_level6/Dung_KhoTriThucLV2'
BANK_DIR = DATA_ROOT/'knowledge_base_level2/reltr_triplet_bank_v1'
OVERRIDES = {'vocab_path': None, 'annotations_dir': None, 'feature_dir': None, 'kb_dir': None}
if SEED not in (42, 44, 46):
    raise ValueError('Thí nghiệm này chỉ tổng hợp seed 42, 44, 46.')
if not DATA_ROOT.is_dir() or not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError('Kiểm tra Drive: cần DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2.')
if not (BANK_DIR/'manifest.json').is_file():
    raise FileNotFoundError('Chưa thấy kho bộ ba. Chạy notebook 01 với cùng BANK_DIR trước.')
print('Warm start:', DATA_ROOT/'baseline_level5/lv5_A/trained_5tokens_v1'/f'seed{SEED}/best_model.pt')
print('Kho bộ ba:', BANK_DIR)
print('Kết quả:', OUTPUT_ROOT/'runs'/f'seed{SEED}')
print('Chấm Test sau train/Val:', RUN_TEST)


Warm start: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed46/best_model.pt
Kho bộ ba: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level2/reltr_triplet_bank_v1
Kết quả: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2/runs/seed46
Chấm Test sau train/Val: True


## 3. Xác minh mã L5A và nạp mã mới
Notebook tự chứa các module mới; các module L5A gốc vẫn đọc từ Drive và đối chiếu SHA-256. Không ghi đè L5.


In [3]:
EXPECTED_SOURCE_HASHES = {'lv5_A/multiseed_5tokens_v1/multiseed.py': '17d8244b2922f8596d646d3472f5045e78f3b589df7418f7ba923162006e0497', 'lv5_A/multiseed_5tokens_v1/storage.py': 'cad0c9d25491f98e04c27de5ed0e1199bc99b7581e6e134e2fcff831684508b1', 'lv5_A/multiseed_5tokens_v1/level5a_pipeline.py': '5e73694a07427e8e251d1f183c96aea94a247c3d8e19f44e61933701dd4df086', 'lv5_A/multiseed_5tokens_v1/level5_model.py': '004e171a7349de30ee8cd453af680cb0dace33420704e13cefcd9d3e5ba2ae05', 'lv5_A/multiseed_5tokens_v1/coco_eval_worker.py': '7d8cf752e85fbab35a6ff275c514990fa81be0d777c895312625df448d39fab0', 'shared_modules/semantic_expansion.py': '126119e4484b8d0419b853f1752321e59a44f8f65544b74d838368ef4f8eb9fe', 'shared_modules/mmr_diversity.py': 'a3f6944129aefc33ad3e57350c93ad9ddd873e0546eaf0575d42dfeea44cf805'}
BASE = DATA_ROOT/'baseline_level5'
for rel, wanted in EXPECTED_SOURCE_HASHES.items():
    path = BASE/rel
    if not path.is_file(): raise FileNotFoundError(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != wanted:
        raise RuntimeError(f'Mã đã thay đổi: {path}. Cần kiểm tra phiên bản trước khi so sánh.')
CODE_DIR = BASE/'lv5_A/multiseed_5tokens_v1'
SHARED_DIR = BASE/'shared_modules'
for name in ['level5a_pipeline','level5_model','storage','semantic_expansion','mmr_diversity']:
    sys.modules.pop(name, None)
for directory in [SHARED_DIR, CODE_DIR]:
    sys.path.insert(0, str(directory))

EMBEDDED_SOURCES = {'drive_io.py': '"""Verified publication for Colab DriveFS; never depends on remote rename.\n\nStage bytes on local disk, copy/read back a recovery copy, then the destination.\nThis is NOT an atomic transaction or a lock across Colab machines. One writer\nper bank/seed is required. Training checkpoints keep their existing journal.\n"""\nimport errno\nimport hashlib\nimport json\nimport os\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef publish(source, target, attempts=4):\n    """Copy from stable local source; retry transient I/O, verify every byte."""\n    source, target = Path(source), Path(target)\n    expected = sha(source)\n    for attempt in range(attempts):\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            try:\n                dest = target.open(\'r+b\')\n            except FileNotFoundError:\n                dest = target.open(\'xb\')\n            with source.open(\'rb\') as inp, dest:\n                dest.seek(0)\n                for block in iter(lambda: inp.read(8 * 1024 * 1024), b\'\'):\n                    dest.write(block)\n                dest.truncate()\n                dest.flush()\n                try:\n                    os.fsync(dest.fileno())\n                except OSError as exc:\n                    if exc.errno not in {errno.EINVAL, errno.ENOTSUP, errno.ENOSYS}:\n                        raise\n            if sha(target) != expected:\n                raise OSError(errno.EIO, \'Read-back checksum mismatch\', str(target))\n            return expected\n        except OSError as exc:\n            if exc.errno not in {errno.ENOENT, errno.EIO, errno.EAGAIN, errno.EEXIST,\n                                 errno.EBUSY, errno.ETIMEDOUT} or attempt == attempts - 1:\n                raise OSError(f\'Could not publish/verify {target}. Check Drive mount, \'\n                              \'free space and other writers; keep existing outputs.\') from exc\n            time.sleep(.5 * (attempt + 1))\n\n\ndef write_bytes(path, data):\n    path = Path(path)\n    with tempfile.TemporaryDirectory(prefix=\'lv6_publish_\') as folder:\n        local = Path(folder)/\'payload\'\n        local.write_bytes(data)\n        publish(local, path.with_name(path.name + \'.pending\'))\n        publish(local, path)\n\n\ndef write_json(path, value):\n    raw = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode(\'utf-8\')\n    write_bytes(path, raw)\n\n\ndef read_json(path):\n    path = Path(path)\n    try:\n        return json.loads(path.read_text(encoding=\'utf-8\'))\n    except (FileNotFoundError, UnicodeError, json.JSONDecodeError) as original:\n        # Read-only fallback: a later verified write can republish the final file.\n        for suffix in (\'.pending\', \'.tmp\'):\n            try:\n                return json.loads(path.with_name(path.name + suffix).read_text(encoding=\'utf-8\'))\n            except (FileNotFoundError, UnicodeError, json.JSONDecodeError):\n                pass\n        raise original\n\n\ndef json_available(path):\n    path = Path(path)\n    return any(path.with_name(path.name + s).is_file() for s in (\'\', \'.pending\', \'.tmp\'))\n\n\ndef check_output(folder):\n    """Small actual write/read check before expensive Colab work; keep a receipt."""\n    import uuid\n    folder = Path(folder)\n    receipt = folder/\'_io_checks\'/(uuid.uuid4().hex + \'.json\')\n    value = {\'method\': \'local_stage_copy_sha256_v1\', \'folder\': str(folder)}\n    write_json(receipt, value)\n    if read_json(receipt) != value:\n        raise IOError(\'Output preflight failed: \' + str(folder))\n    print(\'Drive write/read OK:\', folder)\n', 'reltr_relation_reranking.py': '"""Lv6 prototype: ordered lexical relation matching, not a scene-graph parser."""\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\n_WN = None\n_LEMMATIZER = None\n_ALIASES = {\n    \'man\': \'person\', \'woman\': \'person\', \'boy\': \'person\', \'girl\': \'person\',\n    \'child\': \'person\', \'people\': \'person\', \'men\': \'person\', \'women\': \'person\',\n    \'children\': \'person\', \'bike\': \'bicycle\', \'bikes\': \'bicycle\',\n    \'bicycles\': \'bicycle\', \'dogs\': \'dog\', \'cats\': \'cat\', \'horses\': \'horse\',\n    \'riding\': \'ride\', \'rides\': \'ride\', \'ridden\': \'ride\', \'rode\': \'ride\',\n    \'chasing\': \'chase\', \'chases\': \'chase\', \'chased\': \'chase\',\n    \'holding\': \'hold\', \'holds\': \'hold\', \'held\': \'hold\',\n    \'wearing\': \'wear\', \'wears\': \'wear\', \'wore\': \'wear\',\n    \'sitting\': \'sit\', \'sits\': \'sit\', \'sat\': \'sit\',\n    \'standing\': \'stand\', \'stands\': \'stand\', \'stood\': \'stand\',\n    \'playing\': \'play\', \'plays\': \'play\', \'played\': \'play\',\n    \'eating\': \'eat\', \'eats\': \'eat\', \'ate\': \'eat\',\n    \'carrying\': \'carry\', \'carries\': \'carry\', \'carried\': \'carry\',\n}\n_PHRASE_ALIASES = {\n    \'next to\': (\'beside\', \'alongside\'),\n    \'above\': (\'over\',), \'below\': (\'under\', \'beneath\'),\n}\n\n\ndef configure_wordnet(enabled=True):\n    """Fail explicitly when requested WordNet data are unavailable."""\n    global _WN, _LEMMATIZER\n    if not enabled:\n        _WN, _LEMMATIZER = None, None\n        return\n    from nltk.corpus import wordnet\n    from nltk.stem import WordNetLemmatizer\n    wordnet.ensure_loaded()\n    _WN, _LEMMATIZER = wordnet, WordNetLemmatizer()\n\n\ndef canonical_image_id(value):\n    text = str(value).strip().replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n    if re.fullmatch(r\'\\d+\', text):\n        return str(int(text))\n    match = re.fullmatch(r\'COCO_(?:train|val|test)2014_(\\d+)\\.jpg\', text, re.I)\n    if match:\n        return str(int(match.group(1)))\n    raise ValueError(f\'image_id/filename COCO không hợp lệ: {value!r}\')\n\n\ndef words(text):\n    result = []\n    for token in re.findall(r"[a-z]+(?:\'[a-z]+)?", str(text).lower().replace(\'_\', \' \')):\n        token = _ALIASES.get(token, token)\n        if _LEMMATIZER is not None:\n            token = _LEMMATIZER.lemmatize(token, pos=\'v\')\n            token = _LEMMATIZER.lemmatize(token, pos=\'n\')\n        result.append(_ALIASES.get(token, token))\n    return tuple(result)\n\n\ndef load_reltr_triplets(csv_path, min_conf=0.5):\n    """Accept real image/prediction/score CSV and split-column CSV; retain empty images."""\n    if not 0 <= min_conf <= 1:\n        raise ValueError(\'min_conf phải nằm trong [0,1]\')\n    table, seen = {}, defaultdict(set)\n    stats = {\'rows\': 0, \'no_relation\': 0, \'below_threshold\': 0, \'duplicates\': 0}\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        reader = csv.DictReader(handle)\n        names = {name.strip().lower(): name for name in (reader.fieldnames or [])}\n        def pick(*options):\n            return next((names[name] for name in options if name in names), None)\n        image_col = pick(\'image_id\', \'image\', \'file_name\')\n        pred_col = pick(\'prediction\')\n        sub_col, rel_col, obj_col = pick(\'subject\', \'s\'), pick(\'predicate\', \'relation\', \'rel\', \'p\'), pick(\'object\', \'o\')\n        score_col = pick(\'score\', \'confidence\', \'conf\', \'reltr_score\')\n        if not image_col or not score_col or not (pred_col or (sub_col and rel_col and obj_col)):\n            raise ValueError(f\'CSV thiếu schema RelTR hỗ trợ: {reader.fieldnames}\')\n        for row_number, row in enumerate(reader, 2):\n            try:\n                image_id = canonical_image_id(row[image_col])\n                table.setdefault(image_id, [])\n                stats[\'rows\'] += 1\n                score = float(row[score_col])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError(\'confidence phải hữu hạn và thuộc [0,1]\')\n                if pred_col:\n                    prediction = row[pred_col].strip()\n                    if prediction.lower() == \'no_relation\':\n                        stats[\'no_relation\'] += 1\n                        continue\n                    parts = re.split(r\'\\s+-\\s+\', prediction, maxsplit=2)\n                    if len(parts) != 3:\n                        raise ValueError(\'prediction phải có dạng subject - relation - object\')\n                    subject, predicate, obj = parts\n                else:\n                    subject, predicate, obj = row[sub_col], row[rel_col], row[obj_col]\n                key = tuple(\' \'.join(x.lower().replace(\'_\', \' \').split()) for x in (subject, predicate, obj))\n                if not all(key):\n                    raise ValueError(\'bộ ba có thành phần rỗng\')\n                if score < min_conf:\n                    stats[\'below_threshold\'] += 1\n                    continue\n                if key in seen[image_id]:\n                    stats[\'duplicates\'] += 1\n                    old = next(t for t in table[image_id] if (t[\'subject\'], t[\'predicate\'], t[\'object\']) == key)\n                    old[\'confidence\'] = max(old[\'confidence\'], score)\n                    continue\n                seen[image_id].add(key)\n                table[image_id].append(dict(zip((\'subject\', \'predicate\', \'object\'), key), confidence=score))\n            except (ValueError, TypeError, KeyError, AttributeError) as exc:\n                raise ValueError(f\'{csv_path}, dòng {row_number}: {exc}\') from exc\n    stats[\'images\'] = len(table)\n    stats[\'triplets\'] = sum(map(len, table.values()))\n    return table, stats\n\n\ndef expand_predicate(predicate, max_synonyms=4):\n    """Sorted same-synset lemmas of first verb sense; no troponym-as-synonym expansion."""\n    base = words(predicate)\n    variants = {base}\n    variants.update(words(p) for p in _PHRASE_ALIASES.get(\' \'.join(base), ()))\n    if _WN is not None and len(base) == 1:\n        synsets = _WN.synsets(base[0], pos=_WN.VERB)\n        if synsets:\n            variants.update(words(name) for name in synsets[0].lemma_names())\n    variants.discard(())\n    return (base,) + tuple(sorted(variants - {base})[:max_synonyms])\n\n\ndef phrase_spans(tokens, phrase):\n    size = len(phrase)\n    return [(i, i + size) for i in range(len(tokens) - size + 1)\n            if size and tokens[i:i + size] == phrase]\n\n\ndef compile_triplets(triplets):\n    return [dict(t, subject_tokens=words(t[\'subject\']), object_tokens=words(t[\'object\']),\n                 predicate_variants=expand_predicate(t[\'predicate\'])) for t in triplets]\n\n\ndef relation_score(text, prepared_triplets, max_gap=12):\n    """Ordered S -> P -> O lexical evidence, restricted to one clause. Not full parsing."""\n    matched, total = [], 0.0\n    clauses = re.split(r\'[.!?;]|\\b(?:and|but|while|whereas)\\b\', text.lower())\n    clauses = [words(clause) for clause in clauses]\n    for triplet in prepared_triplets:\n        conf = triplet[\'confidence\']\n        total += conf\n        hit = False\n        for tokens in clauses:\n            if any(t in {\'not\', \'never\', \'without\', \'no\'} or t.endswith("n\'t") for t in tokens):\n                continue\n            ss = phrase_spans(tokens, triplet[\'subject_tokens\'])\n            oo = phrase_spans(tokens, triplet[\'object_tokens\'])\n            pp = [span for variant in triplet[\'predicate_variants\'] for span in phrase_spans(tokens, variant)]\n            hit = any(s1 <= p0 and p1 <= o0 and p0-s1 <= max_gap and o0-p1 <= max_gap\n                      for s0, s1 in ss for p0, p1 in pp for o0, o1 in oo)\n            if hit:\n                break\n        if hit:\n            matched.append(triplet)\n    return (sum(t[\'confidence\'] for t in matched) / total if total else 0.0), [\n        [t[\'subject\'], t[\'predicate\'], t[\'object\']] for t in matched]\n\n\ndef rerank_candidates(candidates, triplets_by_image, alpha=0.2):\n    """Use original normalized L5A baseline_score; alpha=0 preserves scores and stable ranks."""\n    if not 0 <= alpha <= 1:\n        raise ValueError(\'alpha phải nằm trong [0,1]\')\n    groups, seen = defaultdict(list), set()\n    prepared = {}\n    for raw in candidates:\n        item = dict(raw)\n        image_id = canonical_image_id(item[\'image_id\'])\n        caption_id = str(item[\'caption_id\'])\n        if (image_id, caption_id) in seen:\n            raise ValueError(f\'Caption trùng ID trong cùng ảnh: {image_id}/{caption_id}\')\n        seen.add((image_id, caption_id))\n        if not isinstance(item[\'text\'], str) or not item[\'text\'].strip():\n            raise ValueError(\'Caption rỗng/không phải chuỗi\')\n        base = float(item[\'baseline_score\'])\n        if not math.isfinite(base) or not 0 <= base <= 1:\n            raise ValueError(\'baseline_score phải là điểm L5A đã chuẩn hóa trong [0,1]\')\n        if item.get(\'source_image_id\') is not None and canonical_image_id(item[\'source_image_id\']) == image_id:\n            raise ValueError(\'Ứng viên có cùng source_image_id với ảnh truy vấn\')\n        if image_id not in triplets_by_image:\n            raise ValueError(f\'Thiếu bản ghi RelTR cho ảnh {image_id}; kiểm tra split/image_id\')\n        if image_id not in prepared:\n            prepared[image_id] = compile_triplets(triplets_by_image[image_id]) if alpha else []\n        score, matches = relation_score(item[\'text\'], prepared[image_id]) if alpha else (0.0, [])\n        final = base if alpha == 0 or not prepared[image_id] else (1-alpha)*base + alpha*score\n        item.update(image_id=image_id, baseline_score=base, relation_score=score,\n                    final_score=final, matched_triplets=matches)\n        groups[image_id].append(item)\n    for items in groups.values():\n        items.sort(key=lambda item: -item[\'final_score\'])\n    return dict(groups)\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()', 'hybrid_knowledge.py': '"""Train-only RelTR SPO bank. No caption references from Val/Test are embedded."""\nimport hashlib\nimport importlib.metadata\nimport json\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport reltr_relation_reranking as R\nfrom reltr_relation_reranking import load_reltr_triplets\nimport drive_io as IO\n\n# Exact previous implementation reviewed for this storage-only migration.\n# The encoder, inputs, order, thresholds and recipe must still match exactly.\nLEGACY_STORAGE_CODE = \'d8ded22d3ddf094e2f39662bb613b06fba699cb9e2b9ecdab598b019f4f6340e\'\n\n\ndef compatible_bank_code(saved, current=None):\n    return saved in {current or sha(__file__), LEGACY_STORAGE_CODE}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    separators=(\',\', \':\')).encode(\'utf-8\')).hexdigest()\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return IO.read_json(path)\n\n\ndef write_json(path, value):\n    IO.write_json(path, value)\n\n\ndef guard(path, value):\n    if IO.json_available(path):\n        if read_json(path) != value:\n            raise ValueError(\'Existing configuration differs; do not overwrite: \' + str(path))\n    else:\n        write_json(path, value)\n\n\ndef save_array(path, array):\n    path = Path(path)\n    with tempfile.TemporaryDirectory(prefix=\'lv6_array_\') as folder:\n        local = Path(folder)/\'payload.npy\'\n        with local.open(\'wb\') as f:\n            np.save(f, array, allow_pickle=False)\n        checksum = IO.publish(local, path)\n    write_json(path.with_suffix(\'.sha256.json\'), {\'sha256\': checksum})\n\n\ndef bank_source_contract(path, expected):\n    """Preserve the original contract and chunks for the known I/O-only fix."""\n    if not IO.json_available(path):\n        write_json(path, expected)\n        return expected\n    old = read_json(path)\n    if old == expected:\n        return old\n    candidate = dict(old, source_code=expected[\'source_code\'])\n    if old.get(\'source_code\') != LEGACY_STORAGE_CODE or candidate != expected:\n        raise ValueError(\'Bank inputs/recipe/code changed; preserving existing bank: \' + str(path))\n    print(\'Resume bank from known pre-fix writer; all data/recipe hashes unchanged.\')\n    return old\n\n\ndef load_chunk(path):\n    """None means an uncommitted batch; a checksum mismatch still stops the run."""\n    path = Path(path)\n    if not path.is_file():\n        return None\n    try:\n        return checked_array(path)\n    except (FileNotFoundError, UnicodeError, json.JSONDecodeError):\n        print(\'Re-encode incomplete batch checksum:\', path.name)\n        return None\n\n\ndef checked_array(path):\n    path = Path(path)\n    if sha(path) != read_json(path.with_suffix(\'.sha256.json\'))[\'sha256\']:\n        raise IOError(\'Array checksum failed: \' + str(path))\n    return np.load(path, mmap_mode=\'r\', allow_pickle=False)\n\n\ndef split_ids(root, annotations_dir=None):\n    root = Path(root)\n    candidates = ([Path(annotations_dir)] if annotations_dir else [\n        root/\'COCO_2014/annotations_trainval2014/annotations\',\n        root/\'COCO_2014/annotations\', root/\'annotations\', root])\n    candidates = [p for p in candidates if (p/\'dataset_coco.json\').is_file() or\n                  all((p/f\'captions_karpathy_{s}.json\').is_file() for s in (\'train\', \'val\', \'test\'))]\n    if len(candidates) != 1:\n        raise ValueError(\'Set ANNOTATIONS_DIR to the unique Karpathy source: \' + str(candidates))\n    ann = candidates[0]\n    ids = {s: set() for s in (\'train\', \'val\', \'test\')}\n    if (ann/\'dataset_coco.json\').is_file():\n        for row in read_json(ann/\'dataset_coco.json\')[\'images\']:\n            s = \'train\' if row[\'split\'] == \'restval\' else row[\'split\']\n            iid = int(row[\'cocoid\'])\n            if s not in ids or iid in ids[s]:\n                raise ValueError(\'Invalid split or duplicate image ID\')\n            ids[s].add(iid)\n    else:\n        for s in ids:\n            obj = read_json(ann/f\'captions_karpathy_{s}.json\')\n            ids[s] = ({int(r[\'image_id\']) for r in obj[\'annotations\']} if \'annotations\' in obj\n                      else {int(k) for k in obj})\n    if {s: len(v) for s, v in ids.items()} != {\'train\': 113287, \'val\': 5000, \'test\': 5000}:\n        raise ValueError(\'Wrong Karpathy split sizes\')\n    if ids[\'train\'] & ids[\'val\'] or ids[\'train\'] & ids[\'test\'] or ids[\'val\'] & ids[\'test\']:\n        raise ValueError(\'Split leakage\')\n    return {s: sorted(v) for s, v in ids.items()}\n\n\ndef make_triplet_records(table, train_ids):\n    """Dedup exact ordered SPO; preserve EVERY training source for self-exclusion."""\n    train_ids = set(map(int, train_ids))\n    if set(map(int, table)) != train_ids:\n        raise ValueError(\'RelTR Train CSV must cover exactly Train+Restval, including no_relation rows\')\n    merged = {}\n    for iid, triples in table.items():\n        for t in triples:\n            key = (t[\'subject\'], t[\'predicate\'], t[\'object\'])\n            merged.setdefault(key, {})[int(iid)] = float(t[\'confidence\'])\n    result = []\n    for key in sorted(merged):\n        sources = merged[key]\n        result.append(dict(row_id=len(result), subject=key[0], predicate=key[1], object=key[2],\n            text=\' \'.join(key), source_image_ids=sorted(sources),\n            source_confidences=[sources[i] for i in sorted(sources)]))\n    if not result:\n        raise ValueError(\'No valid training triplets after filtering\')\n    return result\n\n\ndef valid_source(record, query_id):\n    return next((i for i in record[\'source_image_ids\'] if i != int(query_id)), None)\n\n\ndef combine_tokens(caption_rows, caption_scores, relation_rows, relation_scores, offset):\n    """Exactly five unique memory rows; 3 caption + up to 2 SPO, then caption fallback."""\n    if len(caption_rows) != 5 or len(set(map(int, caption_rows))) != 5:\n        raise ValueError(\'Need five distinct caption rows\')\n    n = min(2, len(relation_rows))\n    if len(set(map(int, relation_rows[:n]))) != n or len(relation_scores) < n:\n        raise ValueError(\'Invalid triplet rows\')\n    rows = list(map(int, caption_rows[:3])) + [offset + int(r) for r in relation_rows[:n]]\n    scores = list(map(float, caption_scores[:3])) + list(map(float, relation_scores[:n]))\n    rows += list(map(int, caption_rows[3:5-n]))\n    scores += list(map(float, caption_scores[3:5-n]))\n    if len(rows) != 5 or len(set(rows)) != 5 or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid hybrid memory\')\n    if not all(0 <= s <= 1 for s in scores):\n        raise ValueError(\'Memory confidence outside [0,1]\')\n    return rows, scores\n\n\ndef build_bank(root, out, min_confidence=.5, batch_size=128, annotations_dir=None, kb_dir=None):\n    """Encode unique SPO with the SAME frozen CLIP text space as the existing KB."""\n    import torch\n    from transformers import CLIPModel, CLIPTokenizer\n    from huggingface_hub import model_info\n    from tqdm.auto import tqdm\n    root, out = Path(root), Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    ids = split_ids(root, annotations_dir)\n    train_set = set(ids[\'train\'])\n    if kb_dir is None:\n        found = [p for p in [root/\'knowledge_base_level1_karpathy\', root/\'knowledge_base\']\n                 if (p/\'text_embeddings_train_restval.npy\').is_file()]\n        if len(found) != 1:\n            raise ValueError(\'Set KB_DIR; ambiguous/missing caption bank\')\n        kb_dir = found[0]\n    kb_dir = Path(kb_dir)\n    caption_path, map_path = kb_dir/\'text_embeddings_train_restval.npy\', kb_dir/\'text_embedding_mapping.jsonl\'\n    emb = np.load(caption_path, mmap_mode=\'r\')\n    captions, caption_ids = [], []\n    with map_path.open(encoding=\'utf-8\') as f:\n        for i, line in enumerate(f):\n            r = json.loads(line)\n            if int(r.get(\'row_id\', i)) != i or r[\'split\'] not in (\'train\', \'restval\') or int(r[\'image_id\']) not in train_set:\n                raise ValueError(\'Caption mapping contains invalid row or held-out source\')\n            captions.append(str(r[\'caption\']).strip())\n            caption_ids.append(int(r[\'image_id\']))\n    if emb.shape != (len(captions), 512) or set(caption_ids) != train_set or not all(captions):\n        raise ValueError(\'Caption bank/mapping mismatch\')\n    for start in range(0, len(emb), 8192):\n        x = emb[start:start+8192]\n        if not np.isfinite(x).all() or not np.allclose(np.linalg.norm(x, axis=1), 1, atol=1e-3):\n            raise ValueError(\'Caption bank must be finite, L2-normalized\')\n    csv_path = root/\'reltr_inputs/coco_train_reltr_detection.csv\'\n    table, stats = load_reltr_triplets(csv_path, min_confidence)\n    records = make_triplet_records(table, ids[\'train\'])\n    del table\n    source_contract = dict(version=\'hybrid-spo-bank-v1\', source_code=sha(__file__),\n        parser_code=sha(R.__file__),\n        split_sha256={s: digest(v) for s, v in ids.items()}, min_confidence=min_confidence,\n        caption_embeddings_sha256=sha(caption_path), caption_mapping_sha256=sha(map_path),\n        reltr_train_sha256=sha(csv_path), mapping_sha256=digest(records),\n        caption_rows=len(emb), triplet_rows=len(records), template=\'subject predicate object\',\n        dedup=\'ordered_exact_spo_all_sources\', embedding_dim=512, batch_size=batch_size)\n    source_contract = bank_source_contract(out/\'source_contract.json\', source_contract)\n    write_json(out/\'io_implementation.json\', dict(source_code=sha(__file__),\n        drive_io_code=sha(IO.__file__), original_source_code=source_contract[\'source_code\']))\n    manifest_path = out/\'manifest.json\'\n    if IO.json_available(manifest_path):\n        manifest, _, _ = load_bank(out)\n        if manifest[\'source\'] != source_contract:\n            raise ValueError(\'Existing bank has different sources\')\n        if not manifest_path.is_file():\n            write_json(manifest_path, manifest)\n        print(\'Bank already complete:\', out)\n        return manifest\n    model_id = \'openai/clip-vit-base-patch32\'\n    encoder_path = out/\'encoder_contract.json\'\n    revision = read_json(encoder_path)[\'revision\'] if IO.json_available(encoder_path) else model_info(model_id).sha\n    encoder = dict(model=model_id, revision=revision,\n        transformers=importlib.metadata.version(\'transformers\'), torch=torch.__version__,\n        encode=\'text_model.pooler_output -> text_projection -> L2, float32\', max_length=77)\n    guard(encoder_path, encoder)\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    model = CLIPModel.from_pretrained(model_id, revision=revision).float().to(device).eval()\n    tokenizer = CLIPTokenizer.from_pretrained(model_id, revision=revision)\n    for p in model.parameters():\n        p.requires_grad_(False)\n\n    @torch.inference_mode()\n    def encode(texts):\n        inputs = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors=\'pt\').to(device)\n        x = model.text_projection(model.text_model(**inputs).pooler_output)\n        x = torch.nn.functional.normalize(x.float(), dim=-1).cpu().numpy()\n        if x.shape != (len(texts), 512) or not np.isfinite(x).all():\n            raise ValueError(\'CLIP returned invalid embeddings\')\n        return x.astype(np.float32)\n\n    # Verify encoder identity empirically before mixing old and new embeddings.\n    probe = np.linspace(0, len(captions)-1, 32, dtype=int)\n    sim = np.sum(encode([captions[i] for i in probe]) * np.asarray(emb[probe]), axis=1)\n    if float(sim.min()) < .999:\n        raise ValueError(f\'CLIP differs from caption KB: min cosine={sim.min():.6f}; do not mix spaces\')\n    write_json(out/\'encoder_alignment.json\', dict(rows=probe.tolist(), cosines=sim.tolist(), minimum=float(sim.min())))\n    chunks = []\n    for start in tqdm(range(0, len(records), batch_size), desc=\'CLIP training SPO\'):\n        path = out/\'embedding_chunks\'/f\'{start:08d}.npy\'\n        # Uncommitted chunk is safe to re-encode: no model training or Test occurs here.\n        x = load_chunk(path)\n        if x is None:\n            x = encode([r[\'text\'] for r in records[start:start+batch_size]])\n            save_array(path, x)\n        if x.shape != (min(batch_size, len(records)-start), 512):\n            raise ValueError(\'Bad embedding chunk\')\n        chunks.append(np.asarray(x))\n    triplet_emb = np.concatenate(chunks)\n    save_array(out/\'triplet_embeddings.npy\', triplet_emb)\n    write_json(out/\'triplet_mapping.json\', records)\n    manifest = dict(source=source_contract, encoder=encoder, stats=stats,\n        alignment_min_cosine=float(sim.min()), complete=True,\n        triplet_embeddings_sha256=sha(out/\'triplet_embeddings.npy\'),\n        triplet_mapping_sha256=sha(out/\'triplet_mapping.json\'))\n    manifest[\'bank_sha256\'] = digest(manifest)\n    write_json(manifest_path, manifest)\n    print(\'BUILD COMPLETE:\', len(records), \'unique ordered SPO;\', len(emb), \'caption rows\')\n    return manifest\n\n\ndef load_bank(out):\n    out = Path(out)\n    manifest = read_json(out/\'manifest.json\')\n    if not manifest.get(\'complete\') or manifest[\'bank_sha256\'] != digest({k: v for k, v in manifest.items() if k != \'bank_sha256\'}):\n        raise ValueError(\'Incomplete/changed bank manifest\')\n    for name, key in [(\'triplet_embeddings.npy\', \'triplet_embeddings_sha256\'),\n                      (\'triplet_mapping.json\', \'triplet_mapping_sha256\')]:\n        if sha(out/name) != manifest[key]:\n            raise ValueError(\'Bank payload checksum failed: \' + name)\n    embeddings = np.load(out/\'triplet_embeddings.npy\', mmap_mode=\'r\')\n    mapping = read_json(out/\'triplet_mapping.json\')\n    if embeddings.shape != (len(mapping), 512) or len(mapping) != manifest[\'source\'][\'triplet_rows\']:\n        raise ValueError(\'Bank shape mismatch\')\n    if not np.isfinite(embeddings).all() or not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-3):\n        raise ValueError(\'Invalid SPO vectors\')\n    keys = set()\n    for i, r in enumerate(mapping):\n        key = (r[\'subject\'], r[\'predicate\'], r[\'object\'])\n        if r[\'row_id\'] != i or key in keys or not r[\'source_image_ids\'] or r[\'text\'] != \' \'.join(key):\n            raise ValueError(\'Invalid SPO mapping\')\n        keys.add(key)\n    return manifest, embeddings, mapping\n', 'caption_ablation_runner.py': '"""L5A inference ablations on Karpathy Test (or Val), using reviewed native interfaces.\n\nNever calls the original constructor/train/test: all outputs live in baseline_level6.\nFull relevance and Top-5 are computed together once and frozen for all three variants.\nOriginal Top-5 is retained for diagnostics, not mixed into the new comparison.\n"""\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport subprocess\nfrom functools import partial\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport nltk\nimport faiss\nimport level5a_pipeline as P\nimport level5_model as M\nimport semantic_expansion as semantic\nimport mmr_diversity as diversity\nimport storage\n\nFOLDERS = {\'full\': \'_baseline\', \'no_mmr\': \'Tach_Dimersity\', \'no_semantic_expansion\': \'Tach_Sematic\',\n           \'no_reltr_reranking\': \'Tach_ReRanking\'}\nMETRICS = [\'Bleu_1\', \'Bleu_2\', \'Bleu_3\', \'Bleu_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\']\n\n\ndef read_json(path):\n    # Plain read: do not run storage recovery against original L5 folders.\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef read_cache(path, expected):\n    """Validate complete cache or its pending copy, without repairing L5 files."""\n    path = Path(path)\n    failures = []\n    for source in [path, path.with_name(path.name + \'.pending\')]:\n        if not source.is_file():\n            continue\n        try:\n            with np.load(source, allow_pickle=False) as z:\n                meta = json.loads(str(z[\'metadata_json\'].item()))\n                arrays = {k: z[k] for k in z.files if k != \'metadata_json\'}\n            if meta[\'cache_sha256\'] != storage.payload_hash(arrays, meta):\n                raise ValueError(\'Invalid payload hash\')\n        except (ValueError, KeyError, OSError, EOFError) as exc:\n            failures.append(str(exc))\n            continue\n        if any(meta.get(k) != v for k, v in expected.items()):\n            raise ValueError(f\'Cache khác checkpoint/split/config: {source}\')\n        return arrays, meta\n    raise FileNotFoundError(f\'Thiếu cache hoàn chỉnh của L5A: {path}; {failures}\')\n\n\ndef select_variant(labels, captions, embeddings, clip_scores, w, lam, variant):\n    if variant not in (\'no_semantic_expansion\', \'no_reltr_reranking\'):\n        raise ValueError(variant)\n    # Same normalization of label strings as expand_labels, no aliases/synonyms.\n    # Do not add lemmatization that was absent from the actual baseline.\n    if variant == \'no_semantic_expansion\' and labels:\n        terms = {s.lower().strip().replace(\'_\', \' \') for s in labels if s.strip()}\n        jac = np.asarray([semantic.jaccard_semantic(terms, c) for c in captions])\n        raw = w * np.asarray(clip_scores) + (1 - w) * jac\n    else:\n        raw = np.asarray(clip_scores, dtype=np.float64)\n    relevance = P.normalize_scores_per_image(raw)\n    candidates = [dict(position=i, relevance=float(relevance[i]), embedding=e)\n                  for i, e in enumerate(embeddings)]\n    chosen = diversity.mmr_select(candidates, k=5, lambda_param=lam)\n    order = np.asarray([c[\'position\'] for c in chosen], dtype=np.int64)\n    if len(order) != 5 or len(set(order.tolist())) != 5:\n        raise ValueError(\'MMR must select exactly five distinct rows\')\n    order = np.asarray(sorted(order, key=lambda i: (-relevance[i], int(i))), dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef select_no_mmr(relevance):\n    """Remove only MMR, keep precisely the frozen full-baseline relevance."""\n    relevance = np.asarray(relevance, dtype=np.float64)\n    if relevance.ndim != 1 or len(relevance) < 5 or not np.isfinite(relevance).all():\n        raise ValueError(\'Cần ít nhất 5 relevance hữu hạn\')\n    if not ((relevance >= 0) & (relevance <= 1)).all():\n        raise ValueError(\'Relevance phải ở [0,1]\')\n    order = np.asarray(sorted(range(len(relevance)), key=lambda i: (-relevance[i], i))[:5], dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef checked_metrics(metrics):\n    if set(metrics) != set(METRICS) or not all(math.isfinite(float(x)) for x in metrics.values()):\n        raise ValueError(\'Cần đủ 8 metric COCO hữu hạn; không điền metric thiếu bằng 0.\')\n    return {k: float(metrics[k]) for k in METRICS}\n\n\ndef relative_inputs(hashes, root):\n    # Permit moving a whole Data directory; validate the content hashes, not old prefix.\n    prefix = str(root).replace(\'\\\\\', \'/\').rstrip(\'/\') + \'/\'\n    hashes = {k.replace(\'\\\\\', \'/\'): v for k, v in hashes.items()}\n    if not all(k.startswith(prefix) for k in hashes):\n        raise ValueError(\'Input nằm ngoài data_root; cần kiểm tra ánh xạ đường dẫn.\')\n    return {k[len(prefix):]: v for k, v in hashes.items()}\n\n\nclass CaptionAblation(P.Experiment):\n    def __init__(self, data_root, seed, output_root, runner_sha, overrides=None, split="test"):\n        if split not in ("val", "test"):\n            raise ValueError("SPLIT phải là val hoặc test")\n        self.split = split\n        self.root = Path(data_root)\n        self.seed = int(seed)\n        if self.seed not in (42, 43, 44, 45, 46):\n            raise ValueError(\'Invalid seed\')\n        self.cfg = dict(data_root=str(self.root), seed=self.seed, phase=\'ablation\',\n                        eval_timeout_seconds=7200, reltr_dir=str(self.root/\'reltr_inputs\'))\n        self.cfg.update(overrides or {})\n        self.smoke = False\n        self.original = self.root/\'baseline_level5/lv5_A/trained_5tokens_v1\'/f\'seed{self.seed}\'\n        self.output_root = Path(output_root)\n        if self.output_root.resolve().is_relative_to((self.root/\'baseline_level5\').resolve()):\n            raise ValueError(\'Output không được nằm trong baseline_level5\')\n        checkpoint_path = self.original/\'best_model.pt\'\n        log = read_json(self.original/\'training_log.json\')\n        val = read_json(self.original/\'validation_summary.json\')\n        final_cfg = read_json(self.original/\'final_config.json\')\n        run_manifest = read_json(self.original/\'run_manifest.json\')\n        if log.get(\'complete\') is not True or log.get(\'completed_epochs\') != 15 or log.get(\'optimizer_steps\', 0) <= 0:\n            raise ValueError(\'Seed chưa có bằng chứng hoàn tất train 15 epochs\')\n        if val.get(\'trained\') is not True or val.get(\'training_seed\') != self.seed or val.get(\'split\') != \'val\':\n            raise ValueError(\'Validation summary không thuộc seed đã train\')\n        if final_cfg.get(\'selected_on\') != \'validation_only\':\n            raise ValueError(\'Config phải được chọn trên Validation\')\n        self.w, self.lam = float(final_cfg[\'w\']), float(final_cfg[\'lambda_param\'])\n        if self.w != .6 or self.lam not in P.LAMBDA_GRID:\n            raise ValueError(\'Config ngoài recipe L5A\')\n        if (val[\'w\'], val[\'lambda_param\']) != (self.w, self.lam):\n            raise ValueError(\'Summary và final_config khác tham số\')\n        contract_id = val[\'contract_sha256\']\n        if any(obj.get(\'contract_sha256\') != contract_id for obj in [log, final_cfg, run_manifest]):\n            raise ValueError(\'Hồ sơ seed khác contract\')\n        self.code_hashes = {n: P.sha256_file(Path(P.__file__).parent/n) for n in val[\'code_sha256\']}\n        if self.code_hashes != val[\'code_sha256\']:\n            raise ValueError(\'Mã trên Drive đã khác mã tạo baseline; cần kiểm tra trước khi chạy\')\n        shared_hashes = {Path(m.__file__).name: P.sha256_file(m.__file__) for m in [semantic, diversity]}\n        if shared_hashes != val[\'shared_modules_sha256\']:\n            raise ValueError(\'Shared modules khác baseline\')\n        self.environment = dict(python=platform.python_version(), torch=torch.__version__,\n            numpy=np.__version__, nltk=nltk.__version__, wordnet=nltk.corpus.wordnet.get_version(),\n            pycocoevalcap=importlib.metadata.version(\'pycocoevalcap\'), faiss=faiss.__version__)\n        self.environment[\'java\'] = subprocess.run([\'java\', \'-version\'], capture_output=True,\n                                                  text=True, check=True).stderr\n        if \'version "11.\' not in self.environment[\'java\']:\n            raise ValueError(\'Evaluator cần Java 11\')\n        if any(self.environment[k] != val[\'environment\'][k] for k in [\'nltk\',\'wordnet\',\'pycocoevalcap\']):\n            raise ValueError(\'NLTK/WordNet/evaluator khác phiên bản baseline\')\n        self.device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\', self.device, \'| seed:\', self.seed, \'| lambda:\', self.lam)\n        self.load_data()  # Original read-only loader, checks split IDs, source KB, feature alignment.\n        old_root = str(Path(run_manifest[\'init_checkpoint\']).parents[3])\n        if relative_inputs(self.input_hashes, self.root) != relative_inputs(run_manifest[\'input_sha256\'], old_root):\n            raise ValueError(\'Dữ liệu/feature/KB/RelTR khác lần train L5A\')\n        saved = P.load_torch(checkpoint_path)\n        if saved.get(\'contract\') != log_without_history(log):\n            # Verify the exact actual training_contract fields, excluding log-only evidence.\n            raise ValueError(\'Checkpoint best_model.pt không khớp training_log\')\n        evidence = saved[\'evidence\']\n        state_hash = M.state_dict_sha256(saved[\'model_state_dict\'])\n        if state_hash != evidence[\'model_state_sha256\'] or state_hash != val[\'model_state_sha256\'] or state_hash != final_cfg[\'checkpoint_sha256\']:\n            raise ValueError(\'Checkpoint khác model đã chấm Validation\')\n        if not evidence.get(\'trained\') or evidence.get(\'training_seed\') != self.seed or evidence.get(\'completed_epochs\') != 15:\n            raise ValueError(\'Checkpoint không có bằng chứng train đúng seed\')\n        if state_hash == evidence[\'initial_state_sha256\']:\n            raise ValueError(\'Checkpoint còn nguyên trọng số khởi tạo\')\n        P.set_seed(self.seed)\n        self.model, self.greedy, beam = M.create_level5(self.stoi, self.device)\n        self.model.load_state_dict(saved[\'model_state_dict\'], strict=True)\n        self.model.eval()\n        del saved\n        self.beam = partial(beam, beam_size=3, max_len=30, alpha=.6)\n        self.model_hash = state_hash\n        self.source_cache_root = self.original.parent/\'cache\'/contract_id[:20]\n        self.identity = dict(protocol=\'l6-three-inference-ablations-v4\', seed=self.seed, split=self.split,\n            evaluation_ids_sha256=P.digest(self.ids[self.split]),\n            model_state_sha256=state_hash, checkpoint_file_sha256=P.sha256_file(checkpoint_path),\n            original_contract_sha256=contract_id, w=self.w, lambda_param=self.lam,\n            beam=3, max_len=30, alpha=.6, runner_sha256=runner_sha,\n            code_sha256=self.code_hashes, shared_sha256=shared_hashes,\n            input_sha256=relative_inputs(self.input_hashes, self.root), environment=self.environment,\n            device=str(self.device), gpu=torch.cuda.get_device_name(0) if self.device.type==\'cuda\' else None)\n        self.contract_id = P.digest(self.identity)\n        self.base_meta = self.identity\n        self.common_dir = self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True, exist_ok=True)\n        self.cache_root = self.common_dir/\'cache\'\n        self.source_contract_id = contract_id\n        self.pre_reference_identity = dict(self.identity)\n        self.reference_dir = self.common_dir/\'retrieval_reference\'\n        self.test_record_path = self.output_root/\'_test_protocol\'/f\'seed{self.seed}\'/\'test_once.json\'\n        P.write_json(self.common_dir/\'experiment_manifest.json\', self.identity)\n\n    def register_test(self, reference_hash=None, completed=None, summary_path=None):\n        """Freeze this seed\'s Test protocol; resume unfinished work, never retune."""\n        if self.split != \'test\':\n            return\n        if self.test_record_path.is_file():\n            record=read_json(self.test_record_path)\n            if record[\'identity\'] != self.pre_reference_identity:\n                raise ValueError(\'Test đã khóa với checkpoint/cấu hình/môi trường khác. Không tự mở lượt Test mới; kiểm tra test_once.json.\')\n        else:\n            record=dict(identity=self.pre_reference_identity, reference_sha256=None,\n                        completed_variants={}, registered_variants=list(FOLDERS))\n        if reference_hash is not None:\n            if record[\'reference_sha256\'] not in (None,reference_hash):\n                raise ValueError(\'Snapshot Test đã đổi. Khôi phục snapshot đã khóa; không tính lại để chọn kết quả.\')\n            record[\'reference_sha256\']=reference_hash\n        if completed is not None:\n            checksum=P.sha256_file(summary_path)\n            previous=record[\'completed_variants\'].get(completed)\n            if previous is not None and previous != checksum:\n                raise ValueError(\'Kết quả Test hoàn tất đã bị thay đổi\')\n            record[\'completed_variants\'][completed]=checksum\n        record[\'status\']=\'complete\' if set(record[\'completed_variants\'])==set(FOLDERS) else \'in_progress\'\n        P.write_json(self.test_record_path,record)\n\n    def original_top5(self):\n        path = self.source_cache_root/f\'w{self.w:.1f}_lambda{self.lam:.1f}\'/f\'{self.split}.npz\'\n        arrays, _ = read_cache(path, dict(contract_sha256=self.source_contract_id,\n            split=self.split, w=self.w, lambda_param=self.lam, stage=\'rerank_mmr\',\n            ids_sha256=P.digest(self.ids[self.split])))\n        P.validate_top5(arrays, len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'], self.ids[self.split]):\n            raise ValueError(\'Baseline cache image IDs khác split đã chọn\')\n        if np.any(self.kb_ids[arrays[\'row_indices\']] == arrays[\'image_ids\'][:, None]):\n            raise ValueError(\'Baseline cache có caption của cùng ảnh truy vấn\')\n        return arrays\n\n    def original_pool(self):\n        chunks = []\n        for start in range(0, len(self.ids[self.split]), 1000):\n            ids = self.ids[self.split][start:start+1000]\n            chunk, _ = read_cache(self.source_cache_root/\'top50\'/self.split/f\'{start:06d}.npz\',\n                dict(contract_sha256=self.source_contract_id, stage=\'faiss_top50\', split=self.split,\n                     ids_sha256=P.digest(ids)))\n            rows, scores = chunk[\'row_indices\'], chunk[\'clip_scores\']\n            if rows.shape != (len(ids), 50) or scores.shape != rows.shape or not np.isfinite(scores).all():\n                raise ValueError(\'Invalid Top-50 shapes/scores\')\n            if not np.issubdtype(rows.dtype, np.integer) or not ((rows >= 0) & (rows < len(self.kb))).all():\n                raise ValueError(\'Invalid KB rows\')\n            if not np.array_equal(chunk[\'image_ids\'], ids) or any(len(set(r)) != 50 for r in rows):\n                raise ValueError(\'Duplicate/misaligned Top-50\')\n            if np.any(self.kb_ids[rows] == np.asarray(ids)[:, None]):\n                raise ValueError(\'Top-50 chứa cùng source image\')\n            chunks.append(chunk)\n        return {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}\n\n    def frozen_reference(self, pool, original_base):\n        """Original rerank_one unchanged; freeze ALL 50 scores, not only selected five.\n\n        This avoids a no-MMR intervention accidentally changing WordNet expansion.\n        Subsequent notebooks reuse this exact snapshot; a lost/regenerated snapshot\n        changes the result contract, so old captions/metrics cannot be reused.\n        """\n        path = self.reference_dir/\'reference.npz\'\n        pool_hash = storage.payload_hash(pool, {})\n        expected = dict(stage=\'paired_full_relevance_v4\', split=self.split, pool_sha256=pool_hash,\n                        source_contract_sha256=self.source_contract_id,\n                        w=self.w, lambda_param=self.lam)\n        if path.is_file() or path.with_name(path.name+\'.pending\').is_file():\n            arrays, meta = read_cache(path, expected)\n        else:\n            if self.split==\'test\' and read_json(self.test_record_path)[\'reference_sha256\'] is not None:\n                raise FileNotFoundError(\'Mất snapshot Test đã khóa. Khôi phục file reference.npz; không sinh snapshot mới.\')\n            rows, scores, fallback, all_relevance = [], [], [], []\n            for pos, iid in enumerate(pool[\'image_ids\']):\n                r = pool[\'row_indices\'][pos]\n                selected, sc, _, fb, relevance = P.rerank_one(\n                    self.labels[self.split][int(iid)], [self.kb_captions[j] for j in r],\n                    np.asarray(self.kb[r], dtype=np.float32), pool[\'clip_scores\'][pos],\n                    self.w, self.lam, return_relevance=True)\n                rows.append(r[selected]); scores.append(sc); fallback.append(fb)\n                all_relevance.append(relevance)\n                if (pos+1) % 500 == 0: print(\'Đóng băng relevance full:\',pos+1,\'/\',len(pool[\'image_ids\']))\n            arrays = dict(image_ids=pool[\'image_ids\'].copy(),row_indices=np.asarray(rows,dtype=np.int64),\n                scores=np.asarray(scores,dtype=np.float32),is_fallback=np.asarray(fallback,dtype=bool),\n                full_relevance=np.asarray(all_relevance,dtype=np.float64))\n            P.save_cache(path,arrays,expected)\n            arrays, meta = read_cache(path,expected)\n        P.validate_top5(arrays,len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'],pool[\'image_ids\']):\n            raise ValueError(\'Reference image IDs mismatch\')\n        relevance = arrays[\'full_relevance\']\n        if relevance.shape != pool[\'row_indices\'].shape or not np.isfinite(relevance).all() or not ((relevance>=0)&(relevance<=1)).all():\n            raise ValueError(\'Invalid frozen relevance\')\n        for pos,(r5,r50) in enumerate(zip(arrays[\'row_indices\'],pool[\'row_indices\'])):\n            positions={int(r):j for j,r in enumerate(r50)}\n            if not set(r5).issubset(positions): raise ValueError(\'Reference Top-5 outside pool\')\n            actual=np.asarray([relevance[pos,positions[int(r)]] for r in r5],dtype=np.float32)\n            if not np.array_equal(actual,arrays[\'scores\'][pos]): raise ValueError(\'Gate scores differ from frozen relevance\')\n        # Hash only content and expected metadata, not cache creation timestamp.\n        reference_hash=storage.payload_hash(arrays,expected)\n        self.register_test(reference_hash=reference_hash)\n        self.identity=dict(self.pre_reference_identity,retrieval_reference_sha256=reference_hash)\n        self.contract_id=P.digest(self.identity);self.base_meta=self.identity\n        self.common_dir=self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True,exist_ok=True)\n        P.write_json(self.common_dir/\'experiment_manifest.json\',self.identity)\n        P.write_json(self.common_dir/\'retrieval_reference_diagnostics.json\',dict(\n            reference_file=str(path),reference_sha256=reference_hash,\n            baseline_policy=\'Full and no-MMR share frozen relevance computed by original rerank_one\',\n            n_images=len(arrays[\'image_ids\']),\n            changed_top5_vs_original=int(np.any(arrays[\'row_indices\']!=original_base[\'row_indices\'],axis=1).sum()),\n            changed_gate_scores_vs_original=int(np.any(arrays[\'scores\']!=original_base[\'scores\'],axis=1).sum())))\n        return arrays\n\n    def make_variant(self, variant, pool, reference=None):\n        rows, scores = [], []\n        for pos, iid in enumerate(pool[\'image_ids\']):\n            r = pool[\'row_indices\'][pos]\n            if variant == \'no_mmr\':\n                if reference is None: raise ValueError(\'No-MMR requires the full frozen relevance\')\n                selected, sc = select_no_mmr(reference[\'full_relevance\'][pos])\n            else:\n                selected, sc = select_variant(self.labels[self.split][int(iid)],\n                    [self.kb_captions[j] for j in r], np.asarray(self.kb[r], dtype=np.float32),\n                    pool[\'clip_scores\'][pos], self.w, self.lam, variant)\n            rows.append(r[selected]); scores.append(sc)\n            if (pos + 1) % 500 == 0:\n                print(variant, pos + 1, \'/\', len(pool[\'image_ids\']))\n        arrays = dict(image_ids=pool[\'image_ids\'].copy(), row_indices=np.asarray(rows, dtype=np.int64),\n            scores=np.asarray(scores, dtype=np.float32),\n            is_fallback=np.asarray([not self.labels[self.split][int(i)] for i in pool[\'image_ids\']], dtype=bool))\n        P.validate_top5(arrays, len(self.kb))\n        return arrays\n\n    def score(self, variant, cache):\n        self.work = self.output_root/FOLDERS[variant]/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.work.mkdir(parents=True, exist_ok=True)\n        summary_path=self.work/\'summary.json\'\n        if summary_path.is_file():\n            summary=read_json(summary_path)\n            metric=read_json(self.work/\'metrics_coco.json\')\n            if (summary[\'identity\']!=self.identity or summary[\'variant\']!=variant\n                or summary[\'split\']!=self.split or summary[\'n_images\']!=len(self.ids[self.split])):\n                raise ValueError(\'Summary khác cấu hình/split/ảnh đang đánh giá\')\n            checked_metrics(summary[\'metrics_raw\'])\n            if (not metric[\'complete\'] or metric[\'metrics\']!=summary[\'metrics_raw\']\n                or any(summary[\'metrics_x100\'].get(m)!=100*summary[\'metrics_raw\'][m] for m in METRICS)):\n                raise ValueError(\'Metric đã lưu không khớp summary\')\n            self.register_test(completed=variant,summary_path=summary_path)\n            print(\'Dùng kết quả hoàn tất, không sinh/chấm lại:\',variant,self.split)\n            return summary\n        # Persist exact decoder inputs and predictions, with resumable prediction chunks.\n        P.save_cache(self.work/\'top5.npz\', cache, self.meta(variant=variant))\n        preds = self.predictions(self.split, cache, \'beam\', self.model_hash, variant)\n        P.write_json(self.work/\'predictions.json\', preds)\n        metrics = checked_metrics(self.evaluate(preds, self.split, self.work/\'metrics_coco.json\'))\n        summary = dict(identity=self.identity, variant=variant, split=self.split, n_images=len(preds),\n            metrics_raw=metrics, metrics_x100={k: 100*v for k, v in metrics.items()},\n            caption_metrics_computed=True, checkpoint=str(self.original/\'best_model.pt\'),\n            limitation=\'Inference ablation on one trained seed; not retraining or a five-seed aggregate.\',\n            fps_computed=False)\n        P.write_json(self.work/\'summary.json\', summary)\n        self.register_test(completed=variant,summary_path=summary_path)\n        return summary\n\n    def run_ablation(self, variant):\n        if variant not in FOLDERS or variant == \'full\':\n            raise ValueError(variant)\n        self.register_test()\n        base = self.original_top5()\n        pool = self.original_pool()\n        for r5, r50 in zip(base[\'row_indices\'], pool[\'row_indices\']):\n            if not set(r5).issubset(set(r50)):\n                raise ValueError(\'Baseline Top-5 nằm ngoài Top-50\')\n        base = self.frozen_reference(pool,base)\n        variant_cache = self.make_variant(variant, pool,base)\n        self.score(\'full\', base)\n        self.score(variant, variant_cache)\n        return self.comparison()\n\n    def comparison(self):\n        summaries = {}\n        for variant, folder in FOLDERS.items():\n            p = self.output_root/folder/f\'seed{self.seed}\'/self.split/self.contract_id[:20]/\'summary.json\'\n            if p.is_file():\n                summary = read_json(p)\n                if summary[\'identity\'] != self.identity:\n                    raise ValueError(\'Không gộp kết quả khác điều kiện\')\n                checked_metrics(summary[\'metrics_raw\'])\n                if any(summary[\'metrics_x100\'].get(m) != 100*summary[\'metrics_raw\'][m] for m in METRICS):\n                    raise ValueError(\'Summary sai thang metric x100\')\n                summaries[variant] = summary\n        rows = []\n        for variant, s in summaries.items():\n            row = {\'variant\': variant, **s[\'metrics_x100\']}\n            row.update({\'delta_\'+m: s[\'metrics_x100\'][m]-summaries[\'full\'][\'metrics_x100\'][m] for m in METRICS})\n            rows.append(row)\n        out = self.common_dir/\'comparison.csv\'\n        with out.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n            writer = csv.DictWriter(f, fieldnames=[\'variant\']+METRICS+[\'delta_\'+m for m in METRICS])\n            writer.writeheader(); writer.writerows(rows)\n        P.write_json(self.common_dir/\'comparison.json\', dict(identity=self.identity, rows=rows,\n            delta_definition=\'ablation minus full; units are metric x100 points\',\n            available_variants=list(summaries)))\n        print(\'Bảng metric x100; delta = bỏ thành phần - đầy đủ. Số âm là giảm.\')\n        for row in rows:\n            print(row)\n        print(\'Đã lưu:\', out)\n        return rows\n\n\ndef log_without_history(log):\n    extra = {\'history\', \'completed_epochs\', \'complete\', \'optimizer_steps\',\n             \'initial_state_sha256\', \'best_epoch\', \'best_val_cider\'}\n    return {k: v for k, v in log.items() if k not in extra}\n', 'training_stop.py': '"""Deterministic early-stopping decision for the paired hybrid study."""\nimport math\n\n\ndef update(best, stale_epochs, score, epoch, max_epochs, patience, min_delta):\n    """Return (improved, new_stale_count, complete, stop_reason)."""\n    if (not math.isfinite(score) or (best is not None and not math.isfinite(best)) or\n            not 1 <= epoch <= max_epochs or patience < 1 or stale_epochs < 0 or\n            not math.isfinite(min_delta) or min_delta < 0):\n        raise ValueError(\'Invalid early-stopping state\')\n    improved = best is None or score > best + min_delta\n    stale = 0 if improved else stale_epochs + 1\n    early = stale >= patience and epoch < max_epochs\n    complete = early or epoch == max_epochs\n    reason = \'early_stopping\' if early else (\'max_epochs\' if complete else None)\n    return improved, stale, complete, reason\n', 'hybrid_caption_runner.py': '"""Paired L5A warm-start experiment: five captions versus 3 captions + 2 SPO.\n\nBoth arms receive the same maximum epoch budget and early-stop rule. Select checkpoints on Val\nonly, then evaluate each frozen arm once on Test. Never modifies L5 artifacts.\n"""\nimport csv\nimport math\nimport tempfile\nfrom pathlib import Path\n\nimport faiss\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.nn.utils.rnn import pad_sequence\nfrom tqdm.auto import tqdm\n\nimport level5a_pipeline as P\nimport level5_model as M\nimport storage\nimport hybrid_knowledge as H\nimport training_stop as T\nfrom caption_ablation_runner import CaptionAblation, checked_metrics, METRICS\n\nARMS = (\'caption_control\', \'hybrid_3cap_2spo\')\nSTUDY_SEEDS = (42, 44, 46)\n\n# The verified L5 reader still calls these module-level functions while building\n# LV6 caches and prediction chunks. Its original writers rename files on DriveFS.\n# Keep its source files (and their L5 checkpoint hashes) unchanged, but route only\n# this runtime\'s writes through the already verified, rename-free storage module.\nP.write_json = storage.write_json\nP.write_torch = storage.save_torch\nP.save_cache = storage.save_cache\nP.load_cache = storage.load_cache\n\n\ndef cache_hash(cache):\n    return storage.payload_hash({k: cache[k] for k in (\'image_ids\', \'row_indices\', \'scores\')}, {})\n\n\ndef cpu_state(model):\n    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n\n\ndef export_table(path, rows):\n    path = Path(path)\n    storage.write_json(path.with_suffix(\'.json\'), rows)\n    import io\n    buffer = io.StringIO(newline=\'\')\n    writer = csv.DictWriter(buffer, fieldnames=list(rows[0]))\n    writer.writeheader()\n    writer.writerows(rows)\n    storage.write_bytes(path.with_suffix(\'.csv\'), buffer.getvalue().encode(\'utf-8-sig\'))\n\n\nclass HybridExperiment(CaptionAblation):\n    def evaluate(self, records, split, out_path, only_cider=False):\n        """Run the unmodified COCO scorer locally, then publish its outputs to Drive.\n\n        The original subprocess uses os.replace for its result; that operation is\n        reliable on Colab\'s local disk but may fail on a mounted Google Drive.\n        """\n        out_path = Path(out_path)\n        ids = [r[\'image_id\'] for r in records]\n        request = {\'predictions\': records,\n                   \'references\': {str(i): self.caps[split][i] for i in ids},\n                   \'only_cider\': only_cider,\n                   \'evaluator\': \'pycocoevalcap_official_java\',\n                   \'pycocoevalcap_version\': self.environment[\'pycocoevalcap\'],\n                   \'code_sha256\': self.code_hashes[\'coco_eval_worker.py\']}\n        request_id = P.digest(request)\n        if H.IO.json_available(out_path):\n            saved = H.IO.read_json(out_path)\n            if saved.get(\'request_sha256\') != request_id:\n                raise RuntimeError(\'Existing COCO score belongs to a different request: \' + str(out_path))\n            if saved.get(\'complete\'):\n                checked_metrics(saved[\'metrics\']) if not only_cider else None\n                if not out_path.is_file():\n                    H.IO.write_json(out_path, saved)\n                return saved[\'metrics\']\n        with tempfile.TemporaryDirectory(prefix=\'lv6_coco_eval_\') as folder:\n            local = Path(folder)/out_path.name\n            try:\n                metrics = P.Experiment.evaluate(self, records, split, local, only_cider=only_cider)\n            finally:\n                for suffix, remote in (\n                    (\'_eval_input.json\', out_path.with_name(out_path.stem+\'_eval_input.json\')),\n                    (\'.log\', out_path.with_suffix(\'.log\')),\n                    (\'.json\', out_path),\n                ):\n                    artifact = local.with_name(local.stem + suffix) if suffix.startswith(\'_\') else local.with_suffix(suffix)\n                    if artifact.is_file():\n                        H.IO.write_bytes(remote, artifact.read_bytes())\n        return metrics\n\n    def __init__(self, data_root, seed, output_root, bank_dir, source_hashes,\n                 epochs=15, batch_size=64, learning_rate=5e-5,\n                 early_stop_patience=4, early_stop_min_delta=0.0, overrides=None):\n        if (int(seed) not in STUDY_SEEDS or epochs < 1 or batch_size < 1 or\n                not 0 < learning_rate < 1 or early_stop_patience < 1 or\n                not math.isfinite(early_stop_min_delta) or early_stop_min_delta < 0):\n            raise ValueError(\'Invalid training recipe\')\n        root = Path(data_root).resolve()\n        expected_output = (root/\'baseline_level6/Dung_KhoTriThucLV2\').resolve()\n        expected_bank = (root/\'knowledge_base_level2/reltr_triplet_bank_v1\').resolve()\n        if Path(output_root).resolve() != expected_output or Path(bank_dir).resolve() != expected_bank:\n            raise ValueError(\'Output/bank must be in Dung_KhoTriThucLV2 and knowledge_base_level2/reltr_triplet_bank_v1\')\n        if not expected_output.is_dir() or not (expected_bank/\'manifest.json\').is_file():\n            raise FileNotFoundError(\'Create Dung_KhoTriThucLV2 and finish notebook 01 before training\')\n        # Reuse the already-reviewed strict L5 checkpoint/data reader only.\n        super().__init__(data_root, seed, Path(output_root)/\'_l5_reader\',\n                         source_hashes[\'caption_ablation_runner.py\'], overrides, split=\'val\')\n        if self.device.type != \'cuda\':\n            raise RuntimeError(\'Use a Colab GPU for the complete study\')\n        self.study_root = Path(output_root)\n        self.bank_manifest, self.triplet_vectors, self.triplets = H.load_bank(bank_dir)\n        source = self.bank_manifest[\'source\']\n        if not H.compatible_bank_code(source[\'source_code\'], source_hashes[\'hybrid_knowledge.py\']) or source[\'parser_code\'] != source_hashes[\'reltr_relation_reranking.py\']:\n            raise ValueError(\'Bank was built with different source modules\')\n        if source[\'split_sha256\'] != {s: H.digest(self.ids[s]) for s in self.ids}:\n            raise ValueError(\'Bank and L5 use different Karpathy splits\')\n        for filename, key in [(\'text_embeddings_train_restval.npy\', \'caption_embeddings_sha256\'),\n                              (\'text_embedding_mapping.jsonl\', \'caption_mapping_sha256\'),\n                              (\'coco_train_reltr_detection.csv\', \'reltr_train_sha256\')]:\n            matches = [v for p, v in self.input_hashes.items() if Path(p).name == filename]\n            if matches != [source[key]]:\n                raise ValueError(\'Bank and L5 source differ: \' + filename)\n        train_set = set(self.ids[\'train\'])\n        if any(not set(r[\'source_image_ids\']) <= train_set for r in self.triplets):\n            raise ValueError(\'Triplet bank contains non-training sources\')\n        self.triplet_index = faiss.IndexFlatIP(512)\n        self.triplet_index.add(np.array(self.triplet_vectors, dtype=np.float32))\n        self.initial_state = cpu_state(self.model)\n        self.initial_hash = self.model_hash\n        self.recipe = dict(epochs=int(epochs), batch_size=int(batch_size), learning_rate=float(learning_rate),\n            weight_decay=1e-4, cosine_eta_min=1e-6, gradient_clip=1.0,\n            checkpoint_selection=\'greedy_val_CIDEr\', early_stopping=True,\n            early_stop_patience=int(early_stop_patience), early_stop_min_delta=float(early_stop_min_delta),\n            trainable=\'all_L5_captioner_parameters; CLIP_and_RelTR_frozen\',\n            save_every_batches=250, beam=3, max_len=30, alpha=.6)\n        identity = {k: v for k, v in self.identity.items()\n                    if k not in (\'protocol\', \'split\', \'evaluation_ids_sha256\', \'device\', \'gpu\')}\n        identity.update(protocol=\'l6-hybrid-kb-paired-finetune-v1\', recipe=self.recipe,\n            modules=source_hashes, bank_sha256=self.bank_manifest[\'bank_sha256\'],\n            arms=list(ARMS), initial_model_sha256=self.initial_hash,\n            memory=\'5 caption OR 3 caption + 2 ordered SPO; shared Linear(512,256)\',\n            relation_retrieval=\'image CLIP -> train SPO CLIP; exact IP; top50 eligible -> top2\',\n            relation_score=\'minmax cosine within eligible top50, ties->1\',\n            fallback=\'fill missing SPO slots with remaining distinct caption top5\',\n            self_exclusion=\'caption source != query; SPO must have another train source\')\n        self.base_meta = identity\n        self.contract_id = P.digest(identity)\n        self.seed_dir = self.study_root/\'runs\'/f\'seed{self.seed}\'\n        # Fixed seed folder prevents quietly opening new Test configurations.\n        H.guard(self.seed_dir/\'study_contract.json\', identity)\n        self.cache_root = self.seed_dir/\'retrieval_cache\'\n        storage.write_json(self.seed_dir/\'runtime_last_session.json\', dict(device=str(self.device),\n            gpu=torch.cuda.get_device_name(0), environment=self.environment))\n        self.arm_caches = {}\n\n    def top50(self, split):\n        old = self.split\n        try:\n            self.split = split\n            return self.original_pool()\n        except FileNotFoundError:\n            return P.Experiment.top50(self, split)\n        finally:\n            self.split = old\n\n    def caption_cache(self, split):\n        target = self.cache_root/\'caption_snapshot\'/f\'{split}.npz\'\n        expected = self.meta(stage=\'caption_snapshot\', split=split)\n        if storage.cache_available(target):\n            cache, _ = storage.load_cache(target, expected)\n        else:\n            old = self.split\n            try:\n                self.split = split\n                try:\n                    cache = self.original_top5()\n                except FileNotFoundError:\n                    print(\'Computing missing caption cache at final Val lambda:\', split, self.lam)\n                    cache = P.Experiment.cache(self, split, self.w, self.lam)\n            finally:\n                self.split = old\n            storage.save_cache(target, cache, expected)\n        P.validate_top5(cache, len(self.kb))\n        if not np.array_equal(cache[\'image_ids\'], self.ids[split]):\n            raise ValueError(\'Caption snapshot has wrong IDs\')\n        if np.any(self.kb_ids[cache[\'row_indices\']] == cache[\'image_ids\'][:, None]):\n            raise ValueError(\'Caption snapshot leaks own image\')\n        return cache\n\n    def retrieve_relations(self, queries, image_ids):\n        """Expand FAISS search if own-source exclusion leaves too few candidates."""\n        queries = np.array(queries, dtype=np.float32, copy=True)\n        if not np.isfinite(queries).all() or (np.linalg.norm(queries, axis=1) <= 0).any():\n            raise ValueError(\'Invalid image queries\')\n        faiss.normalize_L2(queries)\n        size = min(64, len(self.triplets))\n        distances, positions = self.triplet_index.search(queries, size)\n        results = []\n        for qi, iid in enumerate(image_ids):\n            rows, sims = positions[qi], distances[qi]\n            current = size\n            while True:\n                keep = [(int(r), float(s)) for r, s in zip(rows, sims)\n                        if r >= 0 and H.valid_source(self.triplets[int(r)], iid) is not None]\n                if len(keep) >= 50 or current == len(self.triplets):\n                    break\n                current = min(len(self.triplets), current * 2)\n                dd, rr = self.triplet_index.search(queries[qi:qi+1], current)\n                rows, sims = rr[0], dd[0]\n            keep = sorted(keep, key=lambda x: (-x[1], x[0]))[:50]\n            scores = P.normalize_scores_per_image([s for _, s in keep]) if keep else []\n            picked = keep[:2]\n            results.append(([r for r, _ in picked], list(scores[:2]),\n                            [H.valid_source(self.triplets[r], iid) for r, _ in picked]))\n        return results\n\n    def hybrid_cache(self, split, caption):\n        expected = self.meta(stage=\'hybrid_memory\', split=split, caption_cache_sha256=cache_hash(caption))\n        target = self.cache_root/\'hybrid_memory\'/f\'{split}.npz\'\n        if storage.cache_available(target):\n            cache, _ = storage.load_cache(target, expected)\n        else:\n            chunks = []\n            for start in tqdm(range(0, len(self.ids[split]), 256), desc=f\'Hybrid retrieval {split}\'):\n                ids = self.ids[split][start:start+256]\n                path = target.parent/split/f\'{start:06d}.npz\'\n                cm = dict(expected, start=start, image_ids_sha256=P.digest(ids))\n                if storage.cache_available(path):\n                    chunk, _ = storage.load_cache(path, cm)\n                else:\n                    q = self.features[split][[self.fmap[split][i] for i in ids]]\n                    rr = self.retrieve_relations(q, ids)\n                    rows, scores, counts, provenance = [], [], [], []\n                    for j, (relations, rs, sources) in enumerate(rr):\n                        row, sc = H.combine_tokens(caption[\'row_indices\'][start+j], caption[\'scores\'][start+j],\n                                                   relations, rs, len(self.kb))\n                        rows.append(row); scores.append(sc); counts.append(len(relations))\n                        provenance.append(sources + [-1] * (2-len(sources)))\n                    chunk = dict(image_ids=np.asarray(ids, dtype=np.int64),\n                        row_indices=np.asarray(rows, dtype=np.int64), scores=np.asarray(scores, dtype=np.float32),\n                        relation_count=np.asarray(counts, dtype=np.int64),\n                        relation_source_ids=np.asarray(provenance, dtype=np.int64))\n                    storage.save_cache(path, chunk, cm)\n                chunks.append(chunk)\n            cache = {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}\n            storage.save_cache(target, cache, expected)\n        rows, scores = cache[\'row_indices\'], cache[\'scores\']\n        if rows.shape != scores.shape or rows.shape != (len(self.ids[split]), 5):\n            raise ValueError(\'Hybrid memory must be [N,5]\')\n        if not np.issubdtype(rows.dtype, np.integer) or not ((rows >= 0) & (rows < len(self.kb)+len(self.triplets))).all():\n            raise ValueError(\'Invalid hybrid row indices\')\n        if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():\n            raise ValueError(\'Invalid hybrid confidence\')\n        if any(len(set(r.tolist())) != 5 for r in rows):\n            raise ValueError(\'Duplicate memory rows\')\n        if not np.array_equal(cache[\'image_ids\'], self.ids[split]):\n            raise ValueError(\'Hybrid cache has wrong IDs\')\n        for pos, iid in enumerate(cache[\'image_ids\']):\n            rows = cache[\'row_indices\'][pos]\n            cap_rows = rows[rows < len(self.kb)]\n            if np.any(self.kb_ids[cap_rows] == iid):\n                raise ValueError(\'Hybrid caption leakage\')\n            rel_rows = rows[rows >= len(self.kb)] - len(self.kb)\n            if len(rel_rows) != cache[\'relation_count\'][pos]:\n                raise ValueError(\'Relation count mismatch\')\n            for j, row in enumerate(rel_rows):\n                sid = int(cache[\'relation_source_ids\'][pos, j])\n                if sid == iid or sid not in self.triplets[int(row)][\'source_image_ids\']:\n                    raise ValueError(\'Hybrid triplet provenance/leakage error\')\n        storage.write_json(target.with_suffix(\'.coverage.json\'), dict(images=len(cache[\'image_ids\']),\n            two_spo=int((cache[\'relation_count\']==2).sum()), one_spo=int((cache[\'relation_count\']==1).sum()),\n            zero_spo=int((cache[\'relation_count\']==0).sum()), cache_sha256=cache_hash(cache)))\n        return cache\n\n    def prepare(self, split):\n        if split not in self.arm_caches:\n            caption = self.caption_cache(split)\n            self.arm_caches[split] = {ARMS[0]: caption, ARMS[1]: self.hybrid_cache(split, caption)}\n        return self.arm_caches[split]\n\n    def tensors(self, split, cache, positions):\n        ids = cache[\'image_ids\'][positions]\n        rows = cache[\'row_indices\'][positions]\n        knowledge = np.empty((*rows.shape, 512), dtype=np.float32)\n        mask = rows < len(self.kb)\n        knowledge[mask] = self.kb[rows[mask]]\n        knowledge[~mask] = self.triplet_vectors[rows[~mask]-len(self.kb)]\n        features = np.array(self.features[split][[self.fmap[split][int(i)] for i in ids]], dtype=np.float32)\n        scores = np.array(cache[\'scores\'][positions], dtype=np.float32)\n        return tuple(torch.from_numpy(x).to(self.device) for x in (features, knowledge, scores))\n\n    def train_arm(self, arm):\n        if arm not in ARMS:\n            raise ValueError(arm)\n        self.work = self.seed_dir/arm\n        train_cache, val_cache = self.prepare(\'train\')[arm], self.prepare(\'val\')[arm]\n        contract = dict(study=self.contract_id, arm=arm, recipe=self.recipe,\n            initial_sha256=self.initial_hash, train_cache_sha256=cache_hash(train_cache),\n            val_cache_sha256=cache_hash(val_cache))\n        H.guard(self.work/\'training_contract.json\', contract)\n        journal = storage.Journal(self.work, contract)\n        saved = journal.latest()\n        if saved is None and any((self.work/n).exists() for n in (\'best_model.pt\', \'training_log.json\', \'test_once.json\')):\n            raise RuntimeError(\'Missing resume journal despite training evidence; do not reset\')\n        self.model.load_state_dict(self.initial_state, strict=True)\n        P.set_seed(self.seed)\n        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.recipe[\'learning_rate\'], weight_decay=1e-4)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.recipe[\'epochs\'], eta_min=1e-6)\n        if saved is None:\n            cursor = dict(next_epoch=0, next_batch=0, optimizer_steps=0, history=[],\n                loss_sum_total=0., token_total=0, best_val_cider=None, best_epoch=None,\n                best_model_state_dict=None, epochs_without_improvement=0,\n                training_complete=False, stop_reason=None)\n        else:\n            self.model.load_state_dict(saved[\'model_state_dict\'], strict=True)\n            optimizer.load_state_dict(saved[\'optimizer_state_dict\'])\n            for state in optimizer.state.values():\n                for k, v in state.items():\n                    if torch.is_tensor(v) and k != \'step\':\n                        state[k] = v.to(self.device)\n            scheduler.load_state_dict(saved[\'scheduler_state_dict\'])\n            if saved[\'optimizer_steps\']:\n                optimizer._opt_called = True\n            P.restore_rng(saved[\'rng\'])\n            cursor = {k: v for k, v in saved.items() if k not in (\n                \'model_state_dict\', \'optimizer_state_dict\', \'scheduler_state_dict\', \'rng\', \'generation\', \'contract\')}\n            del saved\n        samples = [(pos, text) for pos, iid in enumerate(train_cache[\'image_ids\']) for text in self.caps[\'train\'][int(iid)]]\n        batch_size = self.recipe[\'batch_size\']\n        batches = math.ceil(len(samples)/batch_size)\n        if not (cursor[\'next_epoch\'] == len(cursor[\'history\']) <= self.recipe[\'epochs\'] and 0 <= cursor[\'next_batch\'] <= batches):\n            raise ValueError(\'Invalid training resume cursor\')\n        if cursor[\'training_complete\'] and cursor[\'stop_reason\'] not in (\'early_stopping\', \'max_epochs\'):\n            raise ValueError(\'Completed training is missing its stop reason\')\n        if (self.work/\'test_once.json\').exists() and not cursor[\'training_complete\']:\n            raise RuntimeError(\'Cannot continue training after opening Test\')\n\n        def persist():\n            journal.save(dict(cursor, model_state_dict=self.model.state_dict(),\n                optimizer_state_dict=optimizer.state_dict(), scheduler_state_dict=scheduler.state_dict(), rng=P.rng_state()))\n\n        if journal.generation < 0:\n            persist()\n        for epoch in range(cursor[\'next_epoch\'], self.recipe[\'epochs\']):\n            if cursor[\'training_complete\']:\n                break\n            self.model.train()\n            permutation = np.random.default_rng(self.seed+epoch).permutation(len(samples))\n            bar = tqdm(range(cursor[\'next_batch\'], batches), initial=cursor[\'next_batch\'], total=batches,\n                       desc=f\'{arm} seed{self.seed} epoch {epoch+1}\')\n            for b in bar:\n                chosen = [samples[i] for i in permutation[b*batch_size:(b+1)*batch_size]]\n                visual, knowledge, scores = self.tensors(\'train\', train_cache, np.asarray([r[0] for r in chosen]))\n                tokens = pad_sequence([self.encode(r[1]) for r in chosen], batch_first=True,\n                                      padding_value=self.stoi[\'<PAD>\']).to(self.device)\n                optimizer.zero_grad(set_to_none=True)\n                logits, gate = self.model(visual, knowledge, scores, tokens[:, :-1])\n                targets = tokens[:, 1:].contiguous()\n                total = F.cross_entropy(logits.reshape(-1, len(self.stoi)), targets.reshape(-1),\n                                        ignore_index=self.stoi[\'<PAD>\'], reduction=\'sum\')\n                count = targets.ne(self.stoi[\'<PAD>\']).sum()\n                loss = total/count.clamp_min(1)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(\'Non-finite loss; preserving last valid journal\')\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1., error_if_nonfinite=True)\n                optimizer.step()\n                cursor[\'optimizer_steps\'] += 1\n                cursor[\'next_batch\'] = b+1\n                cursor[\'loss_sum_total\'] += float(total.item())\n                cursor[\'token_total\'] += int(count.item())\n                bar.set_postfix(loss=cursor[\'loss_sum_total\']/max(1, cursor[\'token_total\']))\n                if (b+1) % self.recipe[\'save_every_batches\'] == 0 or b+1 == batches:\n                    persist()\n            model_hash = M.state_dict_sha256(self.model.state_dict())\n            preds = self.predictions(\'val\', val_cache, \'greedy\', model_hash, f\'epoch{epoch+1:02d}\')\n            metrics = self.evaluate(preds, \'val\', self.work/\'epoch_metrics\'/f\'epoch{epoch+1:02d}.json\', only_cider=True)\n            cider = float(metrics[\'CIDEr\'])\n            if not math.isfinite(cider):\n                raise FloatingPointError(\'Invalid Val CIDEr\')\n            improved, stale, complete, stop_reason = T.update(\n                cursor[\'best_val_cider\'], cursor[\'epochs_without_improvement\'], cider,\n                epoch+1, self.recipe[\'epochs\'], self.recipe[\'early_stop_patience\'],\n                self.recipe[\'early_stop_min_delta\'])\n            if improved:\n                cursor.update(best_val_cider=cider, best_epoch=epoch+1, best_model_state_dict=cpu_state(self.model))\n            cursor[\'epochs_without_improvement\'] = stale\n            cursor[\'history\'].append(dict(epoch=epoch+1, val_CIDEr_x100=cider*100,\n                train_loss=cursor[\'loss_sum_total\']/max(1, cursor[\'token_total\']),\n                lr=optimizer.param_groups[0][\'lr\'], improved=improved))\n            scheduler.step()\n            cursor.update(next_epoch=epoch+1, next_batch=0, loss_sum_total=0., token_total=0,\n                          training_complete=complete, stop_reason=stop_reason)\n            persist()\n            storage.write_json(self.work/\'training_log.json\', {k: v for k, v in cursor.items() if k != \'best_model_state_dict\'})\n            if complete:\n                break\n        if not cursor[\'training_complete\'] or cursor[\'optimizer_steps\'] <= 0 or cursor[\'best_model_state_dict\'] is None:\n            raise ValueError(\'Training has not completed\')\n        self.model.load_state_dict(cursor[\'best_model_state_dict\'], strict=True)\n        model_hash = M.state_dict_sha256(self.model.state_dict())\n        if model_hash == self.initial_hash:\n            raise ValueError(\'Weights did not change from L5\')\n        best = dict(model_state_dict=cursor[\'best_model_state_dict\'], contract=contract,\n            model_state_sha256=model_hash, best_epoch=cursor[\'best_epoch\'],\n            optimizer_steps=cursor[\'optimizer_steps\'], completed_epochs=len(cursor[\'history\']),\n            training_complete=True, stop_reason=cursor[\'stop_reason\'])\n        best_path = self.work/\'best_model.pt\'\n        if best_path.exists():\n            old = storage.load_torch(best_path)\n            if old[\'contract\'] != contract or M.state_dict_sha256(old[\'model_state_dict\']) != model_hash:\n                raise ValueError(\'Existing best model differs from completed training journal\')\n        else:\n            storage.save_torch(best_path, best)\n        storage.write_json(self.work/\'training_log.json\', {k: v for k, v in cursor.items() if k != \'best_model_state_dict\'})\n        print(\'TRAIN COMPLETE:\', arm, \'; epochs run\', len(cursor[\'history\']),\n              \'; selected epoch\', cursor[\'best_epoch\'], \'; reason\', cursor[\'stop_reason\'])\n        return model_hash\n\n    def score_arm(self, arm, split):\n        if arm not in ARMS or split not in (\'val\', \'test\'):\n            raise ValueError(\'Invalid arm/split\')\n        self.work = self.seed_dir/arm\n        best = storage.load_torch(self.work/\'best_model.pt\')\n        if best[\'contract\'][\'study\'] != self.contract_id or best[\'contract\'][\'arm\'] != arm:\n            raise ValueError(\'Wrong best checkpoint\')\n        self.model.load_state_dict(best[\'model_state_dict\'], strict=True)\n        model_hash = M.state_dict_sha256(self.model.state_dict())\n        if (model_hash != best[\'model_state_sha256\'] or best.get(\'training_complete\') is not True or\n                not 1 <= best[\'best_epoch\'] <= best[\'completed_epochs\'] <= self.recipe[\'epochs\'] or\n                best.get(\'stop_reason\') not in (\'early_stopping\', \'max_epochs\')):\n            raise ValueError(\'Invalid training evidence\')\n        if split == \'test\':\n            plan = dict(study=self.contract_id, models={})\n            for branch in ARMS:\n                selected = storage.load_torch(self.seed_dir/branch/\'best_model.pt\')\n                validation = storage.read_json(self.seed_dir/branch/\'val_summary.json\')\n                sh = M.state_dict_sha256(selected[\'model_state_dict\'])\n                if (selected[\'contract\'][\'study\'] != self.contract_id or\n                    selected.get(\'training_complete\') is not True or\n                    not 1 <= selected[\'best_epoch\'] <= selected[\'completed_epochs\'] <= self.recipe[\'epochs\'] or\n                    selected.get(\'stop_reason\') not in (\'early_stopping\', \'max_epochs\') or\n                    validation[\'evaluation\'][\'study\'] != self.contract_id or\n                    validation[\'evaluation\'][\'model_state_sha256\'] != sh or not validation[\'complete\']):\n                    raise ValueError(\'Both arms must finish train/Val before Test\')\n                plan[\'models\'][branch] = sh\n                del selected\n            H.guard(self.seed_dir/\'test_plan.json\', plan)\n        cache = self.prepare(split)[arm]\n        evaluation = dict(study=self.contract_id, arm=arm, split=split, seed=self.seed,\n            model_state_sha256=model_hash, cache_sha256=cache_hash(cache),\n            ids_sha256=P.digest(self.ids[split]), beam=3, max_len=30, alpha=.6)\n        if split == \'test\':\n            val = storage.read_json(self.work/\'val_summary.json\')\n            if val[\'evaluation\'][\'model_state_sha256\'] != model_hash or val[\'evaluation\'][\'study\'] != self.contract_id:\n                raise ValueError(\'Must complete Val with the same frozen model before Test\')\n            H.guard(self.work/\'test_once.json\', evaluation)\n        path = self.work/f\'{split}_summary.json\'\n        if path.exists():\n            summary = storage.read_json(path)\n            if summary[\'evaluation\'] != evaluation or not summary.get(\'complete\'):\n                raise ValueError(\'Existing summary has a different evaluation contract\')\n            if summary[\'summary_sha256\'] != P.digest({k: v for k, v in summary.items() if k != \'summary_sha256\'}):\n                raise ValueError(\'Summary payload changed\')\n            checked_metrics(summary[\'metrics_raw\'])\n            return summary\n        predictions = self.predictions(split, cache, \'beam\', model_hash, f\'final_{split}\')\n        storage.write_json(self.work/f\'{split}_predictions.json\', predictions)\n        metrics = checked_metrics(self.evaluate(predictions, split, self.work/\'metrics\'/f\'{split}.json\'))\n        summary = dict(evaluation=evaluation, complete=True, metrics_raw=metrics,\n                        metrics_x100={k: v*100 for k, v in metrics.items()},\n                        selected_epoch=best[\'best_epoch\'], completed_epochs=best[\'completed_epochs\'],\n                        stop_reason=best[\'stop_reason\'])\n        summary[\'summary_sha256\'] = P.digest(summary)\n        storage.write_json(path, summary)\n        return summary\n\n    def compare(self, split):\n        summaries = [storage.read_json(self.seed_dir/a/f\'{split}_summary.json\') for a in ARMS]\n        rows = []\n        for arm, summary in zip(ARMS, summaries):\n            e = summary[\'evaluation\']\n            if e[\'study\'] != self.contract_id or e[\'arm\'] != arm or e[\'split\'] != split:\n                raise ValueError(\'Cannot combine different studies\')\n            metrics = checked_metrics(summary[\'metrics_raw\'])\n            rows.append(dict(arm=arm, seed=self.seed, split=split,\n                completed_epochs=summary[\'completed_epochs\'], selected_epoch=summary[\'selected_epoch\'],\n                stop_reason=summary[\'stop_reason\'], **{k: metrics[k]*100 for k in METRICS}))\n        rows.append(dict(arm=\'delta_hybrid_minus_caption\', seed=self.seed, split=split,\n                         completed_epochs=\'\', selected_epoch=\'\', stop_reason=\'\',\n                          **{k: rows[1][k]-rows[0][k] for k in METRICS}))\n        export_table(self.seed_dir/f\'{split}_comparison\', rows)\n        return rows\n\n    def run(self, run_test=True):\n        # Both branches are trained and validated before any new Test score is opened.\n        self.prepare(\'train\'); self.prepare(\'val\')\n        for arm in ARMS:\n            self.train_arm(arm)\n            self.score_arm(arm, \'val\')\n        result = {\'val\': self.compare(\'val\')}\n        if run_test:\n            for arm in ARMS:\n                self.score_arm(arm, \'test\')\n            result[\'test\'] = self.compare(\'test\')\n        return result\n\n\ndef aggregate(output_root, seeds=STUDY_SEEDS, split=\'test\'):\n    """Require the declared three paired seeds; never silently aggregate fewer."""\n    root = Path(output_root)\n    if tuple(seeds) != STUDY_SEEDS or split not in (\'val\', \'test\'):\n        raise ValueError(\'Aggregate requires seeds 42,44,46 and a val/test split\')\n    records, signatures = {}, []\n    missing = [str(root/\'runs\'/f\'seed{s}\'/a/f\'{split}_summary.json\') for s in seeds for a in ARMS\n               if not (root/\'runs\'/f\'seed{s}\'/a/f\'{split}_summary.json\').is_file()]\n    if missing:\n        print(\'Aggregate chưa đủ các seed đã khai báo:\', \', \'.join(map(str, seeds)))\n        return None\n    for seed in seeds:\n        folder = root/\'runs\'/f\'seed{seed}\'\n        contract = storage.read_json(folder/\'study_contract.json\')\n        signatures.append({k: contract[k] for k in (\'protocol\', \'recipe\', \'modules\', \'bank_sha256\', \'memory\', \'relation_retrieval\', \'relation_score\', \'fallback\', \'self_exclusion\', \'environment\')})\n        records[seed] = {}\n        for arm in ARMS:\n            s = storage.read_json(folder/arm/f\'{split}_summary.json\')\n            if not s[\'complete\'] or s[\'evaluation\'][\'study\'] != P.digest(contract) or s[\'evaluation\'][\'seed\'] != seed or s[\'evaluation\'][\'arm\'] != arm or s[\'evaluation\'][\'split\'] != split:\n                raise ValueError(\'Invalid aggregate summary\')\n            if s[\'summary_sha256\'] != P.digest({k: v for k, v in s.items() if k != \'summary_sha256\'}):\n                raise ValueError(\'Aggregate summary checksum failed\')\n            records[seed][arm] = checked_metrics(s[\'metrics_raw\'])\n    if any(s != signatures[0] for s in signatures[1:]):\n        raise ValueError(\'Different bank/recipe/software across seeds; cannot pool results\')\n    rows = []\n    for metric in METRICS:\n        old = np.asarray([records[s][ARMS[0]][metric]*100 for s in seeds])\n        new = np.asarray([records[s][ARMS[1]][metric]*100 for s in seeds])\n        delta = new-old\n        rows.append(dict(metric=metric, n=len(seeds), caption_mean=float(old.mean()), caption_std=float(old.std(ddof=1)),\n            hybrid_mean=float(new.mean()), hybrid_std=float(new.std(ddof=1)),\n            paired_delta_mean=float(delta.mean()), paired_delta_std=float(delta.std(ddof=1))))\n    export_table(root/f\'{split}_{len(seeds)}seed_summary\', rows)\n    return rows\n'}
RUNTIME_DIR = Path('/content/lv6_hybrid_runtime')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_HASHES = {n: hashlib.sha256(s.encode('utf-8')).hexdigest() for n, s in EMBEDDED_SOURCES.items()}
for name, source in EMBEDDED_SOURCES.items():
    path = RUNTIME_DIR/name
    path.write_text(source, encoding='utf-8')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == SOURCE_HASHES[name]
    sys.modules.pop(Path(name).stem, None)
sys.path.insert(0, str(RUNTIME_DIR))
print('Runtime modules verified:', list(EMBEDDED_SOURCES))


Runtime modules verified: ['drive_io.py', 'reltr_relation_reranking.py', 'hybrid_knowledge.py', 'caption_ablation_runner.py', 'training_stop.py', 'hybrid_caption_runner.py']


## 4. Kiểm tra checkpoint, kho và nguồn dữ liệu
Checkpoint bắt buộc có hồ sơ hoàn tất 15 epochs L5A. Kho mới phải khớp caption bank, RelTR Train và các split đang dùng.


In [4]:
from hybrid_caption_runner import HybridExperiment, aggregate
from drive_io import check_output
from training_stop import update as early_stop_update
check_output(OUTPUT_ROOT)
assert early_stop_update(0.8, 3, 0.7, 5, 15, 4, 0.0) == (False, 4, True, 'early_stopping')
experiment = HybridExperiment(DATA_ROOT, SEED, OUTPUT_ROOT, BANK_DIR, SOURCE_HASHES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    early_stop_patience=EARLY_STOP_PATIENCE, early_stop_min_delta=EARLY_STOP_MIN_DELTA,
    overrides=OVERRIDES)
print('Study:', experiment.contract_id)
print('Initial L5 state:', experiment.initial_hash)
print('Bank:', experiment.bank_manifest['bank_sha256'])


Drive write/read OK: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2
Device: cuda | seed: 46 | lambda: 0.7
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/vocab.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_train.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_val.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level1_karpathy/text_embeddings_train_restval.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level1_karpathy/text_embedding_mapping.jsonl


Validate KB / FAISS:   0%|          | 0/70 [00:00<?, ?it/s]

SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/train_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/train_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_train_reltr_detection.csv
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/val_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/val_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_val_reltr_detection.csv
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/test_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/test_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_test_reltr_detection.csv
Dữ liệu đã kiểm tra; chưa có số liệu thực nghiệm Level 5.
Study: bdaf763a50db35f383cf0af4e2

## 5. Chạy hai nhánh: Train → Val → Test
Mỗi nhánh dừng sau 4 epoch liên tiếp không tăng Greedy Val CIDEr tốt nhất hoặc khi đạt trần 15 epoch; dùng checkpoint Val tốt nhất. Tự lưu hai bản checkpoint luân phiên mỗi 250 batch và cuối epoch; khôi phục optimizer, scheduler, RNG và vị trí batch. Chạy lại cùng cấu hình để tiếp tục. Kết quả đã hoàn tất được dùng lại. Không đổi seed/recipe để vượt lỗi checksum.


In [5]:
import fcntl
with Path('/tmp') .joinpath(f'lv6-hybrid-seed{SEED}.lock').open('w') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    try:
        results = experiment.run(run_test=RUN_TEST)
    finally:
        fcntl.flock(lock, fcntl.LOCK_UN)


Computing missing caption cache at final Val lambda: train 0.7


Semantic MMR train w=0.6 λ=0.7:   0%|          | 0/114 [00:00<?, ?it/s]

Hybrid retrieval train:   0%|          | 0/443 [00:00<?, ?it/s]

Hybrid retrieval val:   0%|          | 0/20 [00:00<?, ?it/s]

caption_control seed46 epoch 1:   0%|          | 0/8856 [00:00<?, ?it/s]

val greedy:   0%|          | 0/79 [00:00<?, ?it/s]

... (75 outputs left out)

test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /tmp/lv6_coco_eval_ze3vpo18/test.log


test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /tmp/lv6_coco_eval_3imhxtl7/test.log


## 6. Bảng 8 chỉ số và mức chênh lệch
`delta = kho lai − kho caption được fine-tune cùng ngân sách`. Số dương là tăng; chưa đủ căn cứ kết luận ý nghĩa thống kê từ một seed. Không ghép trực tiếp với điểm L4/ablation khác giao thức.


In [6]:
import pandas as pd
from IPython.display import display
for split, rows in results.items():
    print(split.upper(), '| metric ×100')
    display(pd.DataFrame(rows).round(4))
    print('CSV:', experiment.seed_dir/f'{split}_comparison.csv')
for arm in ('caption_control', 'hybrid_3cap_2spo'):
    print(arm, '→', experiment.seed_dir/arm)


VAL | metric ×100


,arm,seed,split,completed_epochs,selected_epoch,stop_reason,Bleu_1,Bleu_2,Bleu_3,Bleu_4,METEOR,ROUGE_L,CIDEr,SPICE
0,caption_control,46,val,12,8,early_stopping,75.0755,58.5392,44.7456,34.1937,27.0127,55.4744,108.6851,19.5740
1,hybrid_3cap_2spo,46,val,12,8,early_stopping,74.9033,58.4517,44.6657,34.1011,26.9025,55.3750,107.9842,19.4902
2,delta_hybrid_minus_caption,46,val,,,,-0.1722,-0.0875,-0.0799,-0.0927,-0.1102,-0.0994,-0.7008,-0.0838


CSV: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2/runs/seed46/val_comparison.csv
TEST | metric ×100


,arm,seed,split,completed_epochs,selected_epoch,stop_reason,Bleu_1,Bleu_2,Bleu_3,Bleu_4,METEOR,ROUGE_L,CIDEr,SPICE
0,caption_control,46,test,12,8,early_stopping,75.1803,58.8876,45.2944,34.8340,27.1962,55.7619,111.4026,19.8845
1,hybrid_3cap_2spo,46,test,12,8,early_stopping,75.1201,58.8173,45.2748,34.7978,27.1070,55.6942,111.1972,19.8946
2,delta_hybrid_minus_caption,46,test,,,,-0.0602,-0.0703,-0.0196,-0.0362,-0.0892,-0.0676,-0.2054,0.0100


CSV: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2/runs/seed46/test_comparison.csv
caption_control → /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2/runs/seed46/caption_control
hybrid_3cap_2spo → /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Dung_KhoTriThucLV2/runs/seed46/hybrid_3cap_2spo


## 7. Xem ví dụ tri thức và caption đã sinh
Đây là dữ liệu của lần chạy thật; các quan hệ là dự đoán RelTR, có thể sai.


In [7]:
import storage
split = 'test' if RUN_TEST else 'val'
hybrid = experiment.prepare(split)['hybrid_3cap_2spo']
predictions = storage.read_json(experiment.seed_dir/'hybrid_3cap_2spo'/f'{split}_predictions.json')
old_predictions = storage.read_json(experiment.seed_dir/'caption_control'/f'{split}_predictions.json')
for pos in range(min(3, len(predictions))):
    print('\nẢnh:', predictions[pos]['image_id'])
    for row in hybrid['row_indices'][pos]:
        if row < len(experiment.kb):
            print('  CAP:', experiment.kb_captions[row])
        else:
            print('  SPO:', experiment.triplets[int(row)-len(experiment.kb)]['text'])
    print('Caption cũ:', old_predictions[pos]['caption'])
    print('Kho lai:', predictions[pos]['caption'])



Ảnh: 42
  CAP: A dog asleep with its head on a shoe
  CAP: A sleeping dog with his face laying on a high top tennis shoe
  CAP: A dog lies on a pillow with its head on a shoe.
  SPO: dog laying on bag
  SPO: dog in shoe
Caption cũ: a dog laying on top of a pile of shoes
Kho lai: a dog laying on top of a pile of shoes

Ảnh: 359
  CAP: Black and white photo of a street with a sign on a pole.
  CAP: A black an white photo of an intersection with a Chicago sign.
  CAP: A picture of a pedestrian sign in a city at dawn.
  SPO: light above street
  SPO: light near bus
Caption cũ: black and white photograph of a traffic light
Kho lai: a black and white photo of a street sign

Ảnh: 636
  CAP: A toilet is shown in an abandoned place.
  CAP: a view of a toilet with the seat open from above
  CAP: A dirty toilet is shown with a black seat.
  SPO: toilet with bowl
  SPO: light on toilet
Caption cũ: a white toilet in a very small room
Kho lai: a white toilet in a very small room


## 8. Tổng hợp 3 seed khi đã đủ kết quả
Cell này tự thông báo nếu chưa đủ 42,44,46; không làm hỏng lượt chạy cá nhân. Std dùng ddof=1, có cả chênh lệch theo cặp seed.


In [8]:
summary = aggregate(OUTPUT_ROOT, seeds=(42, 44, 46), split='test' if RUN_TEST else 'val')
if summary is not None:
    display(pd.DataFrame(summary).round(4))


,metric,n,caption_mean,caption_std,hybrid_mean,hybrid_std,paired_delta_mean,paired_delta_std
0,Bleu_1,3,75.1041,0.1285,75.0520,0.2016,-0.0521,0.0829
1,Bleu_2,3,58.8769,0.1178,58.7544,0.1434,-0.1225,0.0477
2,Bleu_3,3,45.3021,0.0862,45.1915,0.1013,-0.1106,0.0802
3,Bleu_4,3,34.8225,0.0151,34.7040,0.0831,-0.1184,0.0715
4,METEOR,3,27.1680,0.0274,27.1116,0.0380,-0.0564,0.0381
5,ROUGE_L,3,55.7524,0.0257,55.6633,0.0802,-0.0891,0.1017
6,CIDEr,3,111.0320,0.3716,110.9125,0.4044,-0.1195,0.1526
7,SPICE,3,19.8899,0.0191,19.8627,0.0857,-0.0272,0.1047
